<a href="https://colab.research.google.com/github/ApichartNgamrungkij/lab-ai-69/blob/main/AI_RAG_Study_Assistant.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import numpy as np
import pandas as pd

print("Python environment is ready!")
print("NumPy version:", np.__version__)
print("Pandas version:", pd.__version__)

Python environment is ready!
NumPy version: 2.4.6
Pandas version: 2.2.3


In [ ]:
print("NumPy:", np.array([1, 2, 3]))
print("Pandas:")
print(pd.DataFrame({
    "หัวข้อ": ["Chunking", "Embedding", "RAG"],
    "สถานะ": ["รอทำ", "รอทำ", "รอทำ"]
}))

NumPy: [1 2 3]
Pandas:
      หัวข้อ สถานะ
0   Chunking  รอทำ
1  Embedding  รอทำ
2        RAG  รอทำ


In [ ]:
!pip install -q google-genai

In [ ]:
from google import genai
from google.colab import userdata

In [ ]:
client = genai.Client(
    api_key=userdata.get("GeminiAPIKey")
)

print("Gemini API is ready!")

Gemini API is ready!


In [114]:
MODEL = "gemini-3.5-flash-lite"

response = client.models.generate_content(
    model=MODEL,
    contents="สวัสดี ช่วยแนะนำตัวสั้น ๆ เป็นภาษาไทย"
)

print(response.text)

สวัสดีครับ! ผมเป็น AI ผู้ช่วยอัจฉริยะ พร้อมช่วยเหลือและให้ข้อมูลที่คุณต้องการ มีอะไรให้ผมช่วยวันนี้ไหมครับ


In [ ]:
import os, re, json, time
from datetime import date
import numpy as np
import pandas as pd
import plotly.express as px
from plotly.subplots import make_subplots
from google import genai
from google.genai import errors
from google.colab import userdata
pd.set_option("display.max_colwidth", 120)

MODEL = "gemini-3.5-flash-lite"
client = genai.Client(api_key=userdata.get("GEMINI_API_KEY"))

def to_gemini(messages, temperature):
    """แปลงซองเป็นรูปแบบของ Gemini: กฎประจำตัว (system) แยกออกมา ที่เหลือคือบทสนทนา"""
    system = "\n\n".join(m["content"] for m in messages if m["role"] == "system") or None
    contents = [{"role": "user" if m["role"] == "user" else "model", "parts": [{"text": m["content"]}]}
                for m in messages if m["role"] != "system"]
    return contents, {"system_instruction": system, "temperature": temperature}

def call_llm(messages, temperature=0.3):
    """ส่งซองให้เจริญ แล้วรับคำตอบกลับเป็นข้อความ (ถ้าโควตาฟรีเต็ม จะรอแล้วลองใหม่)"""
    contents, config = to_gemini(messages, temperature)
    for wait in [10, 20, 40, 0]:
        try:
            return client.models.generate_content(model=MODEL, contents=contents, config=config).text or ""
        except errors.APIError as e:
            if e.code != 429 or wait == 0:
                raise
            print(f"โควตาเต็ม รอ {wait} วินาที")
            time.sleep(wait)

print(call_llm([{"role": "user", "content": "สวัสดี เจริญ แนะนำตัวหนึ่งประโยค"}]))

In [ ]:

import tiktoken

ENC = tiktoken.get_encoding("o200k_base")
ENC_OLD = tiktoken.get_encoding("cl100k_base")

def count_tokens(text, enc=ENC):
    return len(enc.encode(text))

def count_msgs(messages):
    return sum(count_tokens(m["content"]) for m in messages)

print(count_tokens("สวัสดีครับ ผมชื่อเจริญ"), "token")


In [ ]:
!pip -q install pythainlp

In [ ]:
def show_pieces(text, enc=ENC):
    ids = enc.encode(text)
    pieces = [
        enc.decode_single_token_bytes(i).decode("utf-8", errors="replace")
        for i in ids
    ]
    print(f"{len(ids):>3} token │ " + " | ".join(pieces))


show_pieces("How many days can I borrow a book?")
show_pieces("ยืมหนังสือได้กี่วัน")
show_pieces("ยืมหนังสือได้กี่วัน", ENC_OLD)


from pythainlp.tokenize import word_tokenize

print("ถ้าตัดเป็นคำแบบคน:", word_tokenize("ยืมหนังสือได้กี่วัน"))


pairs = [
    ("How many days can I borrow a book?", "ยืมหนังสือได้กี่วัน"),
    ("The library opens at 7:30 in the morning.", "ห้องสมุดเปิดเวลาเจ็ดโมงครึ่งตอนเช้า"),
    ("I am allergic to peanuts, so please do not recommend snacks with nuts.",
     "ผมแพ้ถั่วลิสง ช่วยอย่าแนะนำขนมที่มีถั่วนะครับ"),
]

rows = [
    {
        "English": en,
        "ไทย": th,
        "tok_en": count_tokens(en),
        "tok_th": count_tokens(th),
        "tok_th_เก่า": count_tokens(th, ENC_OLD)
    }
    for en, th in pairs
]

df_tok = pd.DataFrame(rows)
df_tok["ไทย÷อังกฤษ"] = (df_tok.tok_th / df_tok.tok_en).round(2)

df_tok


In [ ]:
SYSTEM_RULES = """คุณคือ "เจริญ" ผู้ช่วยด้านการเรียน

หน้าที่:
- ช่วยผู้ใช้ทำความเข้าใจบทเรียน
- อธิบายเนื้อหาเป็นภาษาไทยที่เข้าใจง่าย
- ยกตัวอย่างเพื่อให้เห็นภาพเมื่อเหมาะสม
- ตอบโดยอ้างอิงจากเอกสารประกอบการเรียน
- หากไม่มีข้อมูลในเอกสาร ให้แจ้งว่าไม่พบข้อมูล
- ห้ามแต่งข้อมูลหรือสร้างคำตอบขึ้นมาเอง
- ตอบตรงคำถามและเป็นลำดับขั้นตอน
"""

In [ ]:
def build_envelope(
    question,
    system_rules=SYSTEM_RULES,
    tools="",
    memory="",
    cards="",
    history=None
):
    """ประกอบข้อมูลทุกชั้นก่อนส่งให้ AI"""

    system = system_rules

    if tools:
        system += "\n\n[เครื่องมือที่ใช้ได้]\n" + tools

    if memory:
        system += "\n\n[บันทึกความจำ]\n" + memory

    if cards:
        system += "\n\n[ข้อมูลจากเอกสาร]\n" + cards

    return (
        [{"role": "system", "content": system}]
        + (history or [])
        + [{"role": "user", "content": question}]
    )

In [ ]:
layers = {
    "กฎประจำตัว": SYSTEM_RULES,

    "เครื่องมือ": "ขณะนี้ยังไม่มีเครื่องมือเพิ่มเติม",

    "ความจำ": "ขณะนี้ยังไม่มีข้อมูลความจำที่บันทึกไว้",

    "เอกสารประกอบการเรียน": "ขณะนี้ยังไม่มีข้อมูลที่ค้นคืนจากเอกสาร",

    "ประวัติแชท": "ขณะนี้ยังไม่มีประวัติการสนทนา",

    "คำถาม": "ช่วยอธิบายว่า RAG คืออะไร"
}

df_layers = pd.DataFrame([
    {
        "ชั้นข้อมูล": name,
        "เนื้อหา": content,
        "Tokens": count_tokens(content)
    }
    for name, content in layers.items()
])

df_layers

In [ ]:
question = "RAG คืออะไร"

env_A = build_envelope(question)

env_B = build_envelope(
    question,
    cards="RAG คือกระบวนการค้นคืนข้อมูลจากเอกสารที่เกี่ยวข้อง "
          "แล้วนำข้อมูลที่ค้นพบมาเป็นบริบทให้ AI ใช้ตอบคำถาม"
)

print("ซอง A:", len(env_A))
print("ซอง B:", len(env_B))

In [ ]:
print("===== ซอง A =====")
print(env_A[0]["content"])

print("\n===== ซอง B =====")
print(env_B[0]["content"])

In [ ]:
print("===== คำตอบจากซอง A =====")
answer_A = call_llm(env_A, temperature=0)
print(answer_A)

print("\n===== คำตอบจากซอง B =====")
answer_B = call_llm(env_B, temperature=0)
print(answer_B)

In [ ]:
filler = []

for i in range(60):
    filler += [
        {
            "role": "user",
            "content": f"เรื่องที่ {i}: ไดโนเสาร์กินพืชตัวใหญ่"
        },
        {
            "role": "assistant",
            "content": "รับทราบครับ"
        }
    ]

ASK = "เล่านิทานไดโนเสาร์สั้น ๆ 5 ประโยค"

env_thin = build_envelope(ASK)

env_thick = build_envelope(
    ASK,
    history=filler
)

print("ซองบาง:", count_msgs(env_thin), "tokens")
print("ซองหนา:", count_msgs(env_thick), "tokens")

In [ ]:
from datetime import date

def days_between(start, end):
    return (date.fromisoformat(end) - date.fromisoformat(start)).days

TOOLS = {
    "days_between": days_between
}

TOOL_RULE = """คุณสามารถใช้เครื่องมือได้:

days_between(start, end)
ใช้คำนวณจำนวนวันระหว่างวันที่สองวัน
รูปแบบวันที่ YYYY-MM-DD

หากต้องการนับวัน ห้ามคำนวณเอง
ให้ตอบกลับเป็น JSON บรรทัดเดียว เช่น
{"tool": "days_between", "args": {"start": "2026-11-01", "end": "2026-11-15"}}

เมื่อได้รับผลจากเครื่องมือแล้ว
จึงนำผลไปตอบผู้ใช้ตามปกติ
"""

In [ ]:
result = TOOLS["days_between"](
    start="2026-11-01",
    end="2026-11-15"
)

print("ผลจากเครื่องมือ:", result)

In [ ]:
import json
import re
import time

def ask_with_tools(question, today, max_rounds=3):
    messages = build_envelope(
        f"(วันนี้คือ {today}) {question}",
        tools=TOOL_RULE
    )

    for rnd in range(1, max_rounds + 1):
        reply = call_llm(messages, temperature=0)

        print(f"รอบ {rnd}: {reply}")

        m = re.search(r"\{.*\}", reply, re.S)

        try:
            req = json.loads(m.group()) if m else {}
        except json.JSONDecodeError:
            req = {}

        if (
            req.get("tool") not in TOOLS
            or not isinstance(req.get("args"), dict)
        ):
            return reply

        result = TOOLS[req["tool"]](**req["args"])

        print(
            f"ระบบรัน {req['tool']} ได้ผลลัพธ์: {result}"
        )

        messages.append({
            "role": "assistant",
            "content": reply
        })

        messages.append({
            "role": "user",
            "content": f"[ผลจากเครื่องมือ]\n{result}\nกรุณานำผลนี้ไปตอบผู้ใช้"
        })

    return "หยุดการทำงานเนื่องจากใช้เครื่องมือครบจำนวนรอบแล้ว"

In [ ]:
answer = ask_with_tools(
    question="ตั้งแต่วันที่ 2026-11-01 ถึง 2026-11-15 มีกี่วัน",
    today="2026-11-07"
)

print("\nคำตอบสุดท้ายจากเจริญ:")
print(answer)

In [ ]:
first = "สวัสดีเจริญ ตอนนี้ผมกำลังทำโปรเจกต์ AI RAG Study Assistant"

r1 = call_llm(build_envelope(first))
print("รอบ 1:", r1)

r2 = call_llm(
    build_envelope("ผมกำลังทำโปรเจกต์อะไรอยู่?")
)
print("\nรอบ 2 (ไม่ส่งประวัติ):", r2)

history = [
    {"role": "user", "content": first},
    {"role": "assistant", "content": r1}
]

r3 = call_llm(
    build_envelope(
        "ผมกำลังทำโปรเจกต์อะไรอยู่?",
        history=history
    )
)
print("\nรอบ 3 (ส่งประวัติ):", r3)

In [ ]:
# ข้อมูลทดสอบของผู้เรียน
chat = [
    {"role": "user", "content": "ผมชอบคำอธิบายแบบทีละขั้น พร้อมตัวอย่าง"},
    {"role": "assistant", "content": "รับทราบครับ"}
]

# จำลองการสนทนาอื่น ๆ ที่เกิดขึ้นภายหลัง
for i in range(1, 11):
    chat += [
        {"role": "user", "content": f"คำถามทดสอบเรื่องที่ {i}"},
        {"role": "assistant", "content": f"คำตอบทดสอบเรื่องที่ {i}"}
    ]

# วิธีที่ 1: ส่งประวัติทั้งหมด
all_history = chat

# วิธีที่ 2: เก็บเฉพาะ 4 ข้อความล่าสุด
recent_history = chat[-4:]

# วิธีที่ 3: สมุดบันทึกข้างตัว
structured_memory = {
    "รูปแบบการเรียนที่ชอบ": "คำอธิบายทีละขั้น พร้อมตัวอย่าง"
}

print("ประวัติทั้งหมด:", len(all_history), "ข้อความ")
print("ประวัติล่าสุด:", len(recent_history), "ข้อความ")
print("ข้อมูลสำคัญในประวัติทั้งหมด:",
      "ทีละขั้น" in str(all_history))
print("ข้อมูลสำคัญในประวัติล่าสุด:",
      "ทีละขั้น" in str(recent_history))
print("ข้อมูลในสมุดบันทึก:", structured_memory)

In [ ]:
NAIVE_RULES = """
คุณคือเจริญ ผู้ช่วยด้านการเรียน
ตอบทุกคำถามอย่างมั่นใจและครบถ้วน
ห้ามตอบว่าไม่ทราบ
"""

Q_PROJECT = "โปรเจกต์ AI RAG Study Assistant ของเราต้องใช้ Vector Database ชนิดใด และใช้ Embedding Model ชื่ออะไร?"

In [ ]:
rows = []

for temp in [0.0, 1.0]:
    for trial in range(3):
        ans = call_llm(
            build_envelope(
                Q_PROJECT,
                system_rules=NAIVE_RULES
            ),
            temperature=temp
        )

        rows.append({
            "temperature": temp,
            "รอบ": trial + 1,
            "คำตอบ": ans
        })

pd.DataFrame(rows)

In [ ]:
ONE_CARD = """
ข้อมูลโปรเจกต์:
ชื่อ: AI RAG Study Assistant
เป้าหมาย: ผู้ช่วยตอบคำถามจากเอกสารประกอบการเรียน
โมเดลภาษา: Gemini
"""

In [ ]:
QUESTIONS = [
    "โปรเจกต์นี้ชื่ออะไร?",
    "โปรเจกต์นี้ใช้โมเดลภาษาอะไร?",
    "โปรเจกต์นี้ใช้ Vector Database ชนิดใด?"
]

SAFE_RULES = """
คุณคือเจริญ ผู้ช่วยด้านการเรียน
ตอบคำถามโดยอ้างอิงจากข้อมูลที่ได้รับเท่านั้น
หากข้อมูลไม่มีระบุไว้ ให้ตอบว่า "ไม่พบข้อมูลในเอกสาร"
ห้ามเดาหรือแต่งข้อมูลเพิ่มเติม
"""

for name, rules in [
    ("เจริญแบบเดิม", NAIVE_RULES),
    ("เจริญแบบระมัดระวัง", SAFE_RULES)
]:
    print(f"\n===== {name} =====")

    for q in QUESTIONS:
        messages = build_envelope(
            q,
            system_rules=rules,
            cards=ONE_CARD
        )

        ans = call_llm(messages, temperature=0)
        print(f"\nคำถาม: {q}")
        print(f"คำตอบ: {ans}")

In [ ]:
STUDY_DOCUMENT = """
# บทเรียนเรื่อง RAG

## ความหมายของ RAG
RAG ย่อมาจาก Retrieval-Augmented Generation
เป็นกระบวนการค้นคืนข้อมูลจากเอกสารที่เกี่ยวข้อง
แล้วนำข้อมูลที่ค้นพบมาเป็นบริบทให้ AI ใช้ตอบคำถาม

## Chunking
Chunking คือการแบ่งเอกสารขนาดใหญ่
ออกเป็นส่วนย่อย ๆ เพื่อให้ง่ายต่อการค้นหา

## Embedding
Embedding คือการแปลงข้อความเป็นเวกเตอร์ตัวเลข
เพื่อใช้แทนความหมายของข้อความ

## Vector Database
เป็นระบบจัดเก็บเวกเตอร์เพื่อค้นหาข้อมูล
ที่มีความหมายใกล้เคียงกับคำถาม

## Top-k Search
คือการค้นหาและเลือกข้อมูลจำนวน k รายการ
ที่มีความใกล้เคียงกับคำถามมากที่สุด
"""

print("จำนวนตัวอักษร:", len(STUDY_DOCUMENT))
print(STUDY_DOCUMENT)

In [ ]:
def chunk_fixed(text, size=120, overlap=20):
    chunks = []
    start = 0

    while start < len(text):
        chunks.append(text[start:start + size])
        start += size - overlap

    return chunks


fixed_chunks = chunk_fixed(STUDY_DOCUMENT)

print("จำนวน Chunk:", len(fixed_chunks))

for i, chunk in enumerate(fixed_chunks):
    print(f"\n--- Chunk {i+1} ---")
    print(chunk)

In [ ]:
def chunk_by_heading(text):
    cards = []

    for part in re.split(r"\n(?=## )", text.strip()):
        if not part.startswith("## "):
            continue

        lines = part.splitlines()

        cards.append({
            "section": lines[0][3:].strip(),
            "text": "\n".join(lines[1:]).strip()
        })

    return cards


heading_chunks = chunk_by_heading(STUDY_DOCUMENT)

print("จำนวน Chunk:", len(heading_chunks))

for i, card in enumerate(heading_chunks):
    print(f"\n--- Chunk {i+1}: {card['section']} ---")
    print(card["text"])

In [ ]:
for card in heading_chunks:
    card["source"] = "เอกสารบทเรียน RAG"
    card["status"] = "ทดลอง"

for i, card in enumerate(heading_chunks):
    print(f"\n--- Chunk {i+1} ---")
    print("Section:", card["section"])
    print("Source:", card["source"])
    print("Status:", card["status"])
    print("Text:", card["text"])

In [ ]:
!pip -q install sentence-transformers

In [ ]:
from sentence_transformers import SentenceTransformer

EMBED_MODEL_NAME = "BAAI/bge-m3"

embedder = SentenceTransformer(EMBED_MODEL_NAME)

def embed(texts):
    """แปลงข้อความเป็นเวกเตอร์"""
    return embedder.encode(
        list(texts),
        normalize_embeddings=True
    )

print("Embedding Model:", EMBED_MODEL_NAME)
print("โหลดโมเดลสำเร็จ!")

In [ ]:
# นำหัวข้อและเนื้อหามารวมกันก่อนแปลง
chunk_texts = [
    card["section"] + "\n" + card["text"]
    for card in heading_chunks
]

# แปลง Chunk ทั้งหมดเป็นเวกเตอร์
V = embed(chunk_texts)

print("จำนวน Chunk:", len(chunk_texts))
print("ขนาดเวกเตอร์:", V.shape)
print("\nเวกเตอร์ของ Chunk แรก:")
print(V[0])

In [ ]:
# คำนวณความใกล้เคียงของเวกเตอร์ทุกคู่
similarity = V @ V.T

print("ขนาดตาราง:", similarity.shape)
print(similarity)

In [ ]:
def search_chunks(query, k=3):
    # แปลงคำถามเป็นเวกเตอร์
    query_vec = embed([query])[0]

    # คำนวณความใกล้เคียงกับทุก Chunk
    scores = V @ query_vec

    # เรียงจากคะแนนสูงไปต่ำ
    top_indices = np.argsort(-scores)[:k]

    results = []

    for i in top_indices:
        results.append({
            "section": heading_chunks[i]["section"],
            "text": heading_chunks[i]["text"],
            "score": round(float(scores[i]), 3)
        })

    return results


results = search_chunks("ทำยังไงให้ข้อความกลายเป็นตัวเลข?")

for i, result in enumerate(results):
    print(f"\nอันดับ {i+1}")
    print("หัวข้อ:", result["section"])
    print("คะแนน:", result["score"])
    print("เนื้อหา:", result["text"])

In [ ]:
RAG_RULES = """
คุณคือ "เจริญ" ผู้ช่วยด้านการเรียน

กติกาการตอบ:
1. ตอบคำถามโดยอ้างอิงจาก [ข้อมูลจากเอกสาร] เท่านั้น
2. ระบุหัวข้อเอกสารที่นำมาใช้ตอบ
3. หากไม่มีข้อมูลที่ตอบคำถามได้ ให้ตอบว่า
   "ไม่พบข้อมูลในเอกสารที่ได้รับ"
4. ห้ามเดาหรือแต่งข้อมูลเพิ่มเติม
5. ตอบเป็นภาษาไทย เข้าใจง่าย และตรงคำถาม
6. ไม่ต้องทักทายซ้ำทุกครั้ง ให้ตอบเนื้อหาโดยตรง
"""


def format_cards(results):
    return "\n\n".join(
        f"[หัวข้อ: {r['section']}]\n{r['text']}"
        for r in results
    )


print("เตรียมกฎการตอบของเจริญสำเร็จ!")

In [ ]:
def rag_answer(question, k=3, show_cards=True):
    # 1. ค้นหา Chunk ที่เกี่ยวข้อง
    results = search_chunks(question, k=k)

    # 2. แสดงเอกสารที่ค้นพบ
    if show_cards:
        print("เอกสารที่เจริญค้นพบ:")
        for i, card in enumerate(results):
            print(f"\nอันดับ {i+1}: {card['section']}")
            print("คะแนน:", card["score"])

    # 3. จัดรูปแบบ Chunk ให้เป็น Context
    cards_text = format_cards(results)

    # 4. สร้างข้อความส่งให้ Gemini
    messages = build_envelope(
        question,
        system_rules=RAG_RULES,
        cards=cards_text
    )

    # 5. ให้ Gemini สร้างคำตอบ
    answer = call_llm(messages, temperature=0)

    return answer, results


print("สร้างระบบ RAG Answer สำเร็จ!")

In [ ]:
answer, results = rag_answer(
    "Embedding คืออะไร?"
)

print("\nคำตอบจากเจริญ:")
print(answer)

In [ ]:
answer, results = rag_answer(
    "ทำอย่างไรให้คอมพิวเตอร์เข้าใจความหมายของประโยค?"
)

print("\nคำตอบจากเจริญ:")
print(answer)

In [ ]:
answer, results = rag_answer(
    "Vector Database สามารถรองรับข้อมูลได้สูงสุดกี่ล้านรายการ?"
)

print("\nคำตอบจากเจริญ:")
print(answer)

In [ ]:
GOLDEN = [
    {
        "q": "RAG คืออะไร?",
        "kind": "คำถามตรง",
        "expected": "ค้นคืนข้อมูลจากเอกสาร แล้วนำมาเป็นบริบทให้ AI ตอบ",
        "must": "อธิบายการค้นคืนข้อมูลและการนำมาเป็นบริบท",
        "must_not": "ห้ามแต่งรายละเอียดที่ไม่มีในเอกสาร",
        "split": "dev"
    },
    {
        "q": "ทำอย่างไรให้คอมพิวเตอร์เข้าใจความหมายของข้อความ?",
        "kind": "คำถามต่างถ้อยคำ",
        "expected": "ใช้ Embedding แปลงข้อความเป็นเวกเตอร์ตัวเลข",
        "must": "กล่าวถึง Embedding และเวกเตอร์ตัวเลข",
        "must_not": "ห้ามสับสนกับ Chunking",
        "split": "dev"
    },
    {
        "q": "Vector Database มีหน้าที่อะไร?",
        "kind": "คำถามตรง",
        "expected": "จัดเก็บเวกเตอร์เพื่อค้นหาข้อมูลที่มีความหมายใกล้เคียง",
        "must": "กล่าวถึงการจัดเก็บเวกเตอร์และการค้นหาข้อมูล",
        "must_not": "ห้ามแต่งความสามารถที่เอกสารไม่ได้ระบุ",
        "split": "dev"
    },
    {
        "q": "Vector Database รองรับข้อมูลได้สูงสุดกี่ล้านรายการ?",
        "kind": "ไม่มีคำตอบ",
        "expected": "ไม่พบข้อมูลในเอกสาร",
        "must": "แจ้งว่าไม่มีข้อมูล",
        "must_not": "ห้ามแต่งตัวเลข",
        "split": "dev"
    },
    {
        "q": "ใครเป็นผู้คิดค้น RAG?",
        "kind": "ไม่มีคำตอบ",
        "expected": "ไม่พบข้อมูลในเอกสาร",
        "must": "แจ้งว่าไม่มีข้อมูล",
        "must_not": "ห้ามเดาชื่อบุคคล",
        "split": "holdout"
    }
]

DEV = [c for c in GOLDEN if c["split"] == "dev"]
HOLDOUT = [c for c in GOLDEN if c["split"] == "holdout"]

print("ข้อสอบทั้งหมด:", len(GOLDEN))
print("ชุดทดลอง (DEV):", len(DEV))
print("ชุดทดสอบสุดท้าย (HOLDOUT):", len(HOLDOUT))

pd.DataFrame(GOLDEN)

In [ ]:
for case in DEV:
    answer, results = rag_answer(case["q"], show_cards=False)

    case["answer"] = answer
    case["cards"] = format_cards(results)

    print("=" * 50)
    print("คำถาม:", case["q"])
    print("คำตอบจากเจริญ:", answer)

In [ ]:
JUDGE_RULES = """
คุณคือกรรมการตรวจคำตอบของ AI Study Assistant

ให้คะแนน 1-5 ใน 4 ด้าน:
- faithful: คำตอบถูกต้องตามเอกสาร ไม่มีข้อมูลที่เอกสารไม่รองรับ
- complete: ตอบครบตามสิ่งที่โจทย์กำหนด
- relevant: ตรงประเด็นคำถาม
- rules: ไม่ทำสิ่งที่ระบุไว้ในข้อห้าม

ให้พิจารณาความหมาย ไม่ตรวจว่าตรงกับเฉลยทุกตัวอักษร

หากพบข้อความที่ไม่มีหลักฐานจากเอกสาร
ให้ระบุข้อความนั้นใน unsupported

ตอบเป็น JSON เท่านั้น ตามรูปแบบ:
{
  "scores": {
    "faithful": 1,
    "complete": 1,
    "relevant": 1,
    "rules": 1
  },
  "unsupported": [],
  "reason": "เหตุผลประกอบคะแนน"
}
"""


def judge(case):
    user = f"""
คำถาม: {case['q']}

ข้อมูลจากเอกสาร:
{case['cards']}

คำตอบที่คาดหวัง:
{case['expected']}

สิ่งที่ต้องทำ:
{case['must']}

สิ่งที่ห้ามทำ:
{case['must_not']}

คำตอบของเจริญ:
{case['answer']}
"""

    raw = call_llm([
        {"role": "system", "content": JUDGE_RULES},
        {"role": "user", "content": user}
    ], temperature=0)

    data = json.loads(
        re.search(r"\{.*\}", raw, re.S).group()
    )

    data["pass"] = min(data["scores"].values()) >= 4
    return data


print("เตรียมกรรมการ AI เรียบร้อย!")

In [ ]:
rows = []

for case in DEV:
    result = judge(case)
    case["judge"] = result

    rows.append({
        "คำถาม": case["q"],
        **result["scores"],
        "ผ่าน": result["pass"],
        "เหตุผล": result["reason"]
    })

df_judge = pd.DataFrame(rows)
display(df_judge)

In [ ]:
# แก้ True/False ให้ตรงกับผลตรวจของเรา
human = {
    0: True,  # RAG
    1: True,  # Embedding
    2: True,  # Vector Database
    3: True   # ไม่มีข้อมูล
}

agreement = np.mean([
    human[i] == DEV[i]["judge"]["pass"]
    for i in range(len(DEV))
])

print(f"คนกับกรรมการ AI ตัดสินตรงกัน {agreement:.0%}")

for i, case in enumerate(DEV):
    if human[i] != case["judge"]["pass"]:
        print(
            f"ไม่ตรงกันข้อ {i}: "
            f"คน={human[i]}, กรรมการ={case['judge']['pass']}"
        )

In [ ]:
for case in HOLDOUT:
    answer, results = rag_answer(
        case["q"],
        show_cards=False
    )

    case["answer"] = answer
    case["cards"] = format_cards(results)

    print("คำถาม:", case["q"])
    print("คำตอบจากเจริญ:", answer)

    case["judge"] = judge(case)

    print("\nผลกรรมการ:")
    print(case["judge"])

In [ ]:
from google.colab import files

uploaded = files.upload()

print("\nไฟล์ที่อัปโหลด:")
for filename in uploaded.keys():
    print("-", filename)

In [ ]:
# อ่านไฟล์ข้อความที่ผู้ใช้อัปโหลด
filename = list(uploaded.keys())[0]

if filename.endswith(".txt"):
    document_text = uploaded[filename].decode("utf-8")

    print("ชื่อเอกสาร:", filename)
    print("จำนวนตัวอักษร:", len(document_text))
    print("\nเนื้อหาเอกสาร:")
    print(document_text)

else:
    print("ไฟล์ประเภทนี้ยังไม่รองรับ")

In [ ]:
uploaded_chunks = chunk_by_heading(document_text)

for i, card in enumerate(uploaded_chunks):
    print(f"Chunk {i+1}: {card['section']}")
    print(card["text"])
    print("-" * 40)

In [ ]:
def chunk_by_heading(text):
    cards = []

    parts = re.split(r"\n(?=## )", text.strip())

    for part in parts:
        lines = part.splitlines()

        if lines[0].startswith("## "):
            section = lines[0][3:].strip()
            content = "\n".join(lines[1:]).strip()
        else:
            section = "เนื้อหาเอกสาร"
            content = part.strip()

        if content:
            cards.append({
                "section": section,
                "text": content
            })

    return cards


uploaded_chunks = chunk_by_heading(document_text)

for i, card in enumerate(uploaded_chunks):
    print(f"Chunk {i+1}: {card['section']}")
    print(card["text"])
    print("-" * 40)

In [ ]:
for i, card in enumerate(uploaded_chunks):
    card["source"] = filename
    card["chunk_id"] = i

print("ข้อมูล Chunk พร้อม Metadata:")
display(pd.DataFrame(uploaded_chunks))

In [ ]:
uploaded_texts = [
    card["section"] + "\n" + card["text"]
    for card in uploaded_chunks
]

uploaded_vectors = embed(uploaded_texts)

print("จำนวน Chunk:", len(uploaded_chunks))
print("ขนาดเวกเตอร์:", uploaded_vectors.shape)

In [ ]:
def search_uploaded_chunks(query, k=3):
    query_vec = embed([query])[0]

    scores = uploaded_vectors @ query_vec
    top_indices = np.argsort(-scores)[:k]

    results = []

    for i in top_indices:
        results.append({
            "section": uploaded_chunks[i]["section"],
            "text": uploaded_chunks[i]["text"],
            "source": uploaded_chunks[i]["source"],
            "score": round(float(scores[i]), 3)
        })

    return results


# ทดลองค้นหาจากเอกสารที่อัปโหลด
results = search_uploaded_chunks(
    "Python ใช้ทำอะไร?"
)

for result in results:
    print("หัวข้อ:", result["section"])
    print("เอกสาร:", result["source"])
    print("คะแนน:", result["score"])
    print("เนื้อหา:", result["text"])
    print("-" * 40)


In [ ]:
def rag_answer_uploaded(question, k=3):
    results = search_uploaded_chunks(question, k=k)

    cards_text = "\n\n".join(
        f"[หัวข้อ: {r['section']}]\n"
        f"[เอกสาร: {r['source']}]\n"
        f"{r['text']}"
        for r in results
    )

    messages = build_envelope(
        question,
        system_rules=RAG_RULES,
        cards=cards_text
    )

    answer = call_llm(messages, temperature=0)

    return answer, results


answer, results = rag_answer_uploaded(
    "Python ใช้ทำอะไร?"
)

print("คำถาม: Python ใช้ทำอะไร?")
print("\nคำตอบจากเจริญ:")
print(answer)

In [ ]:
answer, results = rag_answer_uploaded(
    "ใครเป็นผู้สร้างภาษา Python?"
)

print("คำถาม: ใครเป็นผู้สร้างภาษา Python?")
print("\nคำตอบจากเจริญ:")
print(answer)

In [ ]:
# คลังเอกสารของเจริญ
ALL_CHUNKS = []
ALL_VECTORS = None

print("สร้างคลังเอกสารเรียบร้อย")
print("จำนวน Chunk:", len(ALL_CHUNKS))

In [ ]:
def add_document(filename, document_text):
    global ALL_CHUNKS, ALL_VECTORS

    # แบ่งเอกสารเป็น Chunk
    new_chunks = chunk_by_heading(document_text)

    # เพิ่ม Metadata
    for i, card in enumerate(new_chunks):
        card["source"] = filename
        card["chunk_id"] = i

    if not new_chunks:
        print("ไม่พบเนื้อหาในเอกสาร")
        return

    # สร้าง Embedding
    new_texts = [
        card["section"] + "\n" + card["text"]
        for card in new_chunks
    ]

    new_vectors = embed(new_texts)

    # เพิ่มเข้าไปในคลังเดิม
    ALL_CHUNKS.extend(new_chunks)

    if ALL_VECTORS is None:
        ALL_VECTORS = new_vectors
    else:
        ALL_VECTORS = np.vstack([ALL_VECTORS, new_vectors])

    print("เพิ่มเอกสารสำเร็จ:", filename)
    print("จำนวน Chunk ที่เพิ่ม:", len(new_chunks))
    print("Chunk ทั้งหมดในคลัง:", len(ALL_CHUNKS))

In [ ]:
add_document(filename, document_text)

In [ ]:
def search_all_documents(query, k=3):
    if not ALL_CHUNKS or ALL_VECTORS is None:
        return []

    query_vec = embed([query])[0]

    scores = ALL_VECTORS @ query_vec
    top_indices = np.argsort(-scores)[:k]

    results = []

    for i in top_indices:
        results.append({
            "section": ALL_CHUNKS[i]["section"],
            "text": ALL_CHUNKS[i]["text"],
            "source": ALL_CHUNKS[i]["source"],
            "score": round(float(scores[i]), 3)
        })

    return results


print("ระบบค้นหาจากคลังเอกสารพร้อมใช้งาน")

In [ ]:
results = search_all_documents("Python ใช้ทำอะไร?")

for result in results:
    print("เอกสาร:", result["source"])
    print("หัวข้อ:", result["section"])
    print("คะแนน:", result["score"])
    print("เนื้อหา:", result["text"])
    print("-" * 40)

In [ ]:
def rag_answer_all(question, k=3):
    results = search_all_documents(question, k=k)

    if not results:
        return "ยังไม่มีเอกสารในคลัง", []

    cards_text = "\n\n".join(
        f"[หัวข้อ: {r['section']}]\n"
        f"[เอกสาร: {r['source']}]\n"
        f"{r['text']}"
        for r in results
    )

    messages = build_envelope(
        question,
        system_rules=RAG_RULES,
        cards=cards_text
    )

    answer = call_llm(messages, temperature=0)

    return answer, results


print("ระบบ RAG สำหรับคลังเอกสารพร้อมใช้งาน")

In [ ]:
answer, results = rag_answer_all("Python ใช้ทำอะไร?")

print("คำตอบจากเจริญ:")
print(answer)

In [ ]:
answer, results = rag_answer_all("ใครเป็นผู้สร้างภาษา Python?")

print("คำตอบจากเจริญ:")
print(answer)

In [ ]:
from google.colab import files

new_uploads = files.upload()

existing_files = {
    card["source"] for card in ALL_CHUNKS
}

for filename, file_bytes in new_uploads.items():

    if filename in existing_files:
        print("ข้ามไฟล์ที่มีอยู่แล้ว:", filename)
        continue

    if filename.lower().endswith(".txt"):
        document_text = file_bytes.decode("utf-8")
        add_document(filename, document_text)

    else:
        print("ยังไม่รองรับไฟล์:", filename)

In [ ]:
!pip -q install pypdf

from pypdf import PdfReader

print("ติดตั้งตัวอ่าน PDF เรียบร้อย")

In [ ]:
from google.colab import files

source_uploads = files.upload()

print("\nไฟล์ที่อัปโหลด:")
for filename in source_uploads.keys():
    print("-", filename)

In [ ]:
from io import BytesIO
from pypdf import PdfReader

pdf_filename = list(source_uploads.keys())[0]

pdf_reader = PdfReader(BytesIO(source_uploads[pdf_filename]))

pdf_text = "\n".join(
    page.extract_text() or ""
    for page in pdf_reader.pages
)

print("ชื่อไฟล์:", pdf_filename)
print("จำนวนหน้า:", len(pdf_reader.pages))
print("จำนวนตัวอักษร:", len(pdf_text))
print("\nตัวอย่างเนื้อหา PDF:")
print(pdf_text[:2000])

In [ ]:
add_document(pdf_filename, pdf_text)

In [ ]:
questions = [
    "Python ใช้ทำอะไร?",
    "ใครเป็นผู้เสนอแนวคิด Perceptron?",
    "การทดสอบ Turing Test คืออะไร?"
]

for question in questions:
    print("\nคำถาม:", question)

    results = search_all_documents(question, k=3)

    for result in results:
        print("เอกสาร:", result["source"])
        print("หัวข้อ:", result["section"])
        print("คะแนน:", result["score"])
        print("เนื้อหา:", result["text"][:300])

In [ ]:
pdf_pages = [
    page.extract_text() or ""
    for page in pdf_reader.pages
]

pdf_text_by_page = "\n\n".join(
    f"## หน้า {i+1}\n{text}"
    for i, text in enumerate(pdf_pages)
    if text.strip()
)

print("จำนวนหน้าที่อ่านได้:", len(pdf_pages))
print("จำนวนตัวอักษร:", len(pdf_text_by_page))
print(pdf_text_by_page[:1000])

In [ ]:
# 1. เก็บ Chunk ที่ไม่ใช่ PDF ประวัติ AI ไว้
keep_indices = [
    i for i, card in enumerate(ALL_CHUNKS)
    if card["source"] != pdf_filename
]

ALL_CHUNKS = [
    ALL_CHUNKS[i] for i in keep_indices
]

ALL_VECTORS = ALL_VECTORS[keep_indices]

# 2. เพิ่ม PDF กลับเข้าไป โดยแบ่งตามหน้า
add_document(pdf_filename, pdf_text_by_page)

# 3. ตรวจสอบคลังเอกสาร
print("\n--- สรุปคลังเอกสาร ---")
print("จำนวน Chunk ทั้งหมด:", len(ALL_CHUNKS))

for source in set(card["source"] for card in ALL_CHUNKS):
    count = sum(1 for card in ALL_CHUNKS if card["source"] == source)
    print(source, ":", count, "Chunks")


In [ ]:
questions = [
    "ใครเป็นผู้เสนอแนวคิด Perceptron?",
    "การทดสอบ Turing Test คืออะไร?",
    "Python ใช้ทำอะไร?"
]

for question in questions:
    print("\nคำถาม:", question)

    results = search_all_documents(question, k=3)

    for result in results:
        print("เอกสาร:", result["source"])
        print("หัวข้อ:", result["section"])
        print("คะแนน:", result["score"])
        print("เนื้อหา:", result["text"][:400])
        print("-" * 40)

In [ ]:
answer, results = rag_answer_all(
    "การทดสอบ Turing Test คืออะไร?",
    k=3
)

print("คำตอบจากเจริญ:")
print(answer)

In [ ]:
from google.colab import files
from io import BytesIO
from pypdf import PdfReader

def read_uploaded_file(filename, file_bytes):

    # อ่าน TXT
    if filename.lower().endswith(".txt"):
        return file_bytes.decode("utf-8-sig")

    # อ่าน PDF แยกตามหน้า
    elif filename.lower().endswith(".pdf"):
        reader = PdfReader(BytesIO(file_bytes))

        pages = []

        for i, page in enumerate(reader.pages):
            text = page.extract_text() or ""

            if text.strip():
                pages.append(f"## หน้า {i+1}\n{text}")

        return "\n\n".join(pages)

    else:
        return None


# เริ่มอัปโหลดหลายไฟล์
new_uploads = files.upload()

# ตรวจสอบชื่อไฟล์ที่มีอยู่แล้ว
existing_files = {
    card["source"] for card in ALL_CHUNKS
}

for filename, file_bytes in new_uploads.items():

    if filename in existing_files:
        print("ข้ามไฟล์ที่มีอยู่แล้ว:", filename)
        continue

    document_text = read_uploaded_file(filename, file_bytes)

    if document_text is None:
        print("ไม่รองรับไฟล์:", filename)
        continue

    if not document_text.strip():
        print("ไม่พบข้อความในไฟล์:", filename)
        continue

    add_document(filename, document_text)

    existing_files.add(filename)

print("\nอัปโหลดเอกสารเสร็จสิ้น")
print("จำนวนเอกสาร:", len(existing_files))
print("จำนวน Chunk:", len(ALL_CHUNKS))


In [ ]:
# ตรวจสอบ Chunk ของเอกสารใหม่
for source in sorted(set(card["source"] for card in ALL_CHUNKS)):
    print("\n" + "=" * 60)
    print("เอกสาร:", source)

    source_chunks = [
        card for card in ALL_CHUNKS
        if card["source"] == source
    ]

    print("จำนวน Chunk:", len(source_chunks))

    for card in source_chunks[:2]:
        print("\nหัวข้อ:", card["section"])
        print("เนื้อหา:", card["text"][:500])

In [ ]:
test_questions = [
    "การประเมินโมเดลควรดูค่าอะไรบ้าง?",
    "ข้อมูลสกปรกมีปัญหาอะไรบ้าง?",
    "Gradient Descent ทำงานอย่างไร?",
    "Backpropagation คืออะไร?",
    "Python ใช้ทำอะไร?"
]

for question in test_questions:
    print("\n" + "=" * 60)
    print("คำถาม:", question)

    results = search_all_documents(question, k=3)

    for i, result in enumerate(results):
        print(f"\nอันดับ {i+1}")
        print("เอกสาร:", result["source"])
        print("หน้า/หัวข้อ:", result["section"])
        print("คะแนน:", result["score"])
        print("เนื้อหา:", result["text"][:350])

In [ ]:
test_questions = [
    "Gradient Descent ทำงานอย่างไร?",
    "Backpropagation คืออะไร?",
    "การประเมินโมเดลควรดูค่าอะไรบ้าง?"
]

for question in test_questions:
    print("\n" + "=" * 60)
    print("คำถาม:", question)

    results = search_all_documents(question, k=5)

    for i, result in enumerate(results):
        print(f"\nอันดับ {i+1}")
        print("เอกสาร:", result["source"])
        print("หน้า:", result["section"])
        print("คะแนน:", result["score"])
        print("เนื้อหา:", result["text"][:250])

In [ ]:
# แบ่ง Chunk ย่อยจาก Chunk เดิม
SMALL_CHUNKS = []

CHUNK_SIZE = 700
OVERLAP = 100

for card in ALL_CHUNKS:
    text = card["text"]
    start = 0
    part = 0

    while start < len(text):
        end = start + CHUNK_SIZE

        small_card = {
            "section": card["section"],
            "text": text[start:end],
            "source": card["source"],
            "chunk_id": f"{card['chunk_id']}_{part}"
        }

        SMALL_CHUNKS.append(small_card)

        start += CHUNK_SIZE - OVERLAP
        part += 1

# แทนที่คลังเดิม
ALL_CHUNKS = SMALL_CHUNKS

# สร้าง Embedding ใหม่ให้ตรงกับ Chunk ใหม่
all_texts = [
    card["section"] + "\n" + card["text"]
    for card in ALL_CHUNKS
]

ALL_VECTORS = embed(all_texts)

print("แบ่ง Chunk ใหม่เรียบร้อย")
print("จำนวน Chunk เดิม: 35")
print("จำนวน Chunk ใหม่:", len(ALL_CHUNKS))
print("ขนาด Embedding:", ALL_VECTORS.shape)

In [ ]:
test_questions = [
    "การประเมินโมเดลควรดูค่าอะไรบ้าง?",
    "ข้อมูลสกปรกมีปัญหาอะไรบ้าง?",
    "Gradient Descent ทำงานอย่างไร?",
    "Backpropagation คืออะไร?",
    "Python ใช้ทำอะไร?"
]

for question in test_questions:
    print("\n" + "=" * 60)
    print("คำถาม:", question)

    results = search_all_documents(question, k=3)

    for i, result in enumerate(results):
        print(f"\nอันดับ {i+1}")
        print("เอกสาร:", result["source"])
        print("หน้า:", result["section"])
        print("คะแนน:", result["score"])
        print("เนื้อหา:", result["text"][:350])

In [ ]:
questions = [
    "Gradient Descent ทำงานอย่างไร?",
    "Backpropagation คืออะไร?",
    "ข้อมูลสกปรกมีปัญหาอะไรบ้าง?"
]

for question in questions:
    print("\n" + "=" * 60)
    print("คำถาม:", question)

    answer, results = rag_answer_all(question, k=3)

    print("\nคำตอบจากเจริญ:")
    print(answer)

In [ ]:
RAG_RULES = """
คุณคือ "เจริญ" ผู้ช่วยด้านการเรียน

กติกาการตอบ:
1. ตอบคำถามโดยอ้างอิงจาก [ข้อมูลจากเอกสาร] เท่านั้น
2. ห้ามเพิ่มข้อเท็จจริง ตัวเลข หรือรายละเอียดที่ไม่มีในเอกสาร
3. รักษาหน่วยและความหมายของข้อมูลให้ตรงกับต้นฉบับ
4. หากข้อมูลจากเอกสารไม่เพียงพอ ให้บอกว่า
   "ไม่พบข้อมูลเพียงพอในเอกสารที่ได้รับ"
5. หากใช้ข้อมูลจากหลายหน้า ให้ระบุเลขหน้าทุกหน้าที่นำมาใช้
6. แยกให้ชัดเจนระหว่างข้อมูลที่เอกสารระบุ
   กับข้อสรุปที่อนุมานจากข้อมูล
7. ตอบเป็นภาษาไทย เข้าใจง่าย และตรงคำถาม
8. ไม่ต้องทักทายซ้ำทุกครั้ง
"""

In [ ]:
test_questions = [
    "ข้อมูลสกปรกในใบงานมีปัญหาอะไรบ้าง?",
    "เอกสารระบุว่า KNeighborsClassifier เกิด ValueError เพราะค่า NaN ใช่ไหม?"
]

for question in test_questions:
    print("\n" + "=" * 60)
    print("คำถาม:", question)

    answer, results = rag_answer_all(question, k=3)

    print("\nคำตอบจากเจริญ:")
    print(answer)

In [104]:
GOLDEN_TEST = [
    # คำถามที่มีคำตอบในเอกสาร
    {
        "q": "Python ใช้ทำอะไร?",
        "kind": "คำถามตรง",
        "expected": "เป็นภาษาโปรแกรมที่ใช้พัฒนาโปรแกรม",
        "must": "กล่าวว่า Python เป็นภาษาโปรแกรมที่ใช้พัฒนาโปรแกรม",
        "must_not": "ห้ามแต่งความสามารถที่ไม่มีในเอกสาร",
        "split": "dev"
    },
    {
        "q": "ตัวแปรใน Python มีไว้ทำอะไร?",
        "kind": "คำถามตรง",
        "expected": "ใช้สำหรับเก็บข้อมูล",
        "must": "กล่าวถึงการเก็บข้อมูล",
        "must_not": "ห้ามแต่งชนิดข้อมูลที่เอกสารไม่ได้ระบุ",
        "split": "dev"
    },
    {
        "q": "Function มีประโยชน์อย่างไร?",
        "kind": "คำถามตรง",
        "expected": "รวมชุดคำสั่งเพื่อเรียกใช้งาน",
        "must": "กล่าวถึงการรวมชุดคำสั่งและเรียกใช้งาน",
        "must_not": "ห้ามแต่งรายละเอียดเพิ่มเติม",
        "split": "dev"
    },
    {
        "q": "ภาษา Python เอาไว้สร้างอะไร?",
        "kind": "คำถามต่างถ้อยคำ",
        "expected": "ใช้พัฒนาโปรแกรม",
        "must": "กล่าวถึงการพัฒนาโปรแกรม",
        "must_not": "ห้ามแต่งประเภทโปรแกรมที่ไม่มีในเอกสาร",
        "split": "dev"
    },
    {
        "q": "Gradient Descent คืออะไร?",
        "kind": "คำถามตรง",
        "expected": "การเดินสวนทางความชันทีละก้าว",
        "must": "กล่าวถึงการเดินสวนทางความชัน",
        "must_not": "ห้ามแต่งขั้นตอนที่ไม่มีหลักฐาน",
        "split": "dev"
    },
    {
        "q": "ตัวอย่าง Gradient Descent ในใบงานใช้ฟังก์ชันอะไร?",
        "kind": "คำถามตรง",
        "expected": "ความสูงคือ x คูณ x และความชันคือ 2x",
        "must": "กล่าวถึง x คูณ x และความชัน 2x",
        "must_not": "ห้ามเปลี่ยนสูตร",
        "split": "dev"
    },
    {
        "q": "Backpropagation ช่วยเรื่องอะไร?",
        "kind": "คำถามตรง",
        "expected": "ช่วยให้ฝึกโครงข่ายประสาทหลายชั้นได้จริง",
        "must": "กล่าวถึงการฝึกโครงข่ายประสาทหลายชั้น",
        "must_not": "ห้ามแต่งกลไกที่เอกสารไม่ได้อธิบาย",
        "split": "dev"
    },
    {
        "q": "ใครเผยแพร่อัลกอริทึม Backpropagation ตามเอกสาร?",
        "kind": "คำถามตรง",
        "expected": "Rumelhart, Hinton และ Williams",
        "must": "ระบุชื่อทั้งสามคน",
        "must_not": "ห้ามเพิ่มชื่อบุคคลอื่น",
        "split": "dev"
    },
    {
        "q": "ใครเป็นผู้เสนอแนวคิด Perceptron?",
        "kind": "คำถามตรง",
        "expected": "Frank Rosenblatt ในปี 1957",
        "must": "กล่าวถึง Frank Rosenblatt และปี 1957",
        "must_not": "ห้ามสับสนกับผู้คิดค้นแนวคิดอื่น",
        "split": "dev"
    },
    {
        "q": "Turing Test ใช้วัดอะไร?",
        "kind": "คำถามตรง",
        "expected": "ใช้ทดสอบว่าแยกแยะเครื่องจักรจากมนุษย์ผ่านการสนทนาได้หรือไม่",
        "must": "กล่าวถึงการสนทนาและการแยกแยะมนุษย์กับเครื่องจักร",
        "must_not": "ห้ามอธิบายเกินข้อมูลในเอกสาร",
        "split": "dev"
    },
    {
        "q": "ข้อมูล wine_dirty.csv มีค่าหายและแถวซ้ำเท่าไร?",
        "kind": "คำถามตรง",
        "expected": "ค่าหาย 62 ค่า และแถวซ้ำ 10 แถว",
        "must": "ระบุ 62 ค่าและ 10 แถว พร้อมหน่วยให้ถูกต้อง",
        "must_not": "ห้ามสลับหน่วย",
        "split": "dev"
    },
    {
        "q": "wine_dirty.csv มีขนาดเท่าไร?",
        "kind": "คำถามตรง",
        "expected": "188 แถว 15 คอลัมน์",
        "must": "ระบุ 188 แถวและ 15 คอลัมน์",
        "must_not": "ห้ามสลับจำนวนแถวกับคอลัมน์",
        "split": "dev"
    },
    {
        "q": "ทำไม KNeighborsClassifier จึงใช้ข้อมูล wine_dirty.csv โดยตรงไม่ได้?",
        "kind": "คำถามตรง",
        "expected": "เพราะข้อมูลมีค่า NaN ซึ่งโมเดลไม่ยอมรับ",
        "must": "กล่าวถึง NaN และข้อจำกัดของโมเดล",
        "must_not": "ห้ามแต่งสาเหตุอื่น",
        "split": "dev"
    },
    {
        "q": "AlexNet มีความสำคัญอย่างไรต่อประวัติ AI?",
        "kind": "คำถามตรง",
        "expected": "ชนะการแข่งขัน ImageNet ในปี 2012 และเป็นจุดเปลี่ยนของ Deep Learning",
        "must": "กล่าวถึง AlexNet, ImageNet และปี 2012",
        "must_not": "ห้ามแต่งผลการแข่งขัน",
        "split": "dev"
    },
    {
        "q": "เอกสารระบุจำนวนพารามิเตอร์ของ GPT-4 ไว้เท่าไร?",
        "kind": "ไม่มีคำตอบ",
        "expected": "เอกสารระบุว่าตัวเลขไม่ได้เปิดเผยอย่างเป็นทางการ",
        "must": "บอกว่าไม่มีการเปิดเผยตัวเลขอย่างเป็นทางการ",
        "must_not": "ห้ามเดาตัวเลข",
        "split": "dev"
    },

    # Holdout: เก็บไว้ทดสอบหลังปรับระบบ
    {
        "q": "ใครเป็นผู้สร้างภาษา Python ตามบทเรียนเรื่อง Python.txt?",
        "kind": "ไม่มีคำตอบ",
        "expected": "ไม่พบข้อมูลในเอกสาร",
        "must": "แจ้งว่าไม่มีข้อมูล",
        "must_not": "ห้ามเดาชื่อบุคคล",
        "split": "holdout"
    },
    {
        "q": "บทเรียนเรื่อง Python.txt ระบุว่า Python เวอร์ชันล่าสุดคือเวอร์ชันอะไร?",
        "kind": "ไม่มีคำตอบ",
        "expected": "ไม่พบข้อมูลในเอกสาร",
        "must": "แจ้งว่าไม่มีข้อมูล",
        "must_not": "ห้ามเดาหมายเลขเวอร์ชัน",
        "split": "holdout"
    },
    {
        "q": "เอกสารระบุ Learning Rate ที่ใช้ในการทดลอง Gradient Descent เป็นเท่าไร?",
        "kind": "ไม่มีคำตอบ",
        "expected": "ไม่พบข้อมูลเพียงพอในเอกสาร",
        "must": "ไม่แต่งค่าตัวเลขหากไม่มีหลักฐาน",
        "must_not": "ห้ามเดาค่า Learning Rate",
        "split": "holdout"
    },
    {
        "q": "เอกสารระบุว่า wine_dirty.csv ใช้พื้นที่กี่ MB?",
        "kind": "ไม่มีคำตอบ",
        "expected": "ไม่พบข้อมูลในเอกสาร",
        "must": "แจ้งว่าไม่มีข้อมูล",
        "must_not": "ห้ามเดาขนาดไฟล์",
        "split": "holdout"
    },
    {
        "q": "Turing Test ใช้เวลาทดสอบกี่นาทีตามเอกสาร?",
        "kind": "ไม่มีคำตอบ",
        "expected": "ไม่พบข้อมูลในเอกสาร",
        "must": "แจ้งว่าไม่มีข้อมูล",
        "must_not": "ห้ามเดาระยะเวลา",
        "split": "holdout"
    }
]

DEV_TEST = [c for c in GOLDEN_TEST if c["split"] == "dev"]
HOLDOUT_TEST = [c for c in GOLDEN_TEST if c["split"] == "holdout"]

print("Golden Dataset พร้อมแล้ว")
print("คำถามทั้งหมด:", len(GOLDEN_TEST))
print("Dev:", len(DEV_TEST))
print("Holdout:", len(HOLDOUT_TEST))

Golden Dataset พร้อมแล้ว
คำถามทั้งหมด: 20
Dev: 15
Holdout: 5


In [105]:
retrieval_results = []

for i, item in enumerate(DEV_TEST, start=1):
    results = search_all_documents(item["q"], k=3)

    top_result = results[0] if results else None

    retrieval_results.append({
        "ข้อ": i,
        "คำถาม": item["q"],
        "เอกสารอันดับ 1": top_result["source"] if top_result else "ไม่พบ",
        "หัวข้อ": top_result["section"] if top_result else "ไม่พบ",
        "คะแนน": top_result["score"] if top_result else 0,
        "ข้อความที่ค้นพบ": top_result["text"][:180] if top_result else "ไม่พบ"
    })

retrieval_df = pd.DataFrame(retrieval_results)

pd.set_option("display.max_colwidth", 180)
display(retrieval_df)

,ข้อ,คำถาม,เอกสารอันดับ 1,หัวข้อ,คะแนน,ข้อความที่ค้นพบ
0,1,Python ใช้ทำอะไร?,บทเรียนเรื่อง Python.txt,เนื้อหาเอกสาร,0.721,บทเรียนเรื่อง Python\r\n\r\nPython เป็นภาษาโปรแกรมที่ใช้พัฒนาโปรแกรม\r\nตัวแปรใช้สำหรับเก็บข้อมูล\r\nFunction ใช้สำหรับรวมชุดคำสั่งเพื่อเรียกใช้งาน
1,2,ตัวแปรใน Python มีไว้ทำอะไร?,บทเรียนเรื่อง Python.txt,เนื้อหาเอกสาร,0.733,บทเรียนเรื่อง Python\r\n\r\nPython เป็นภาษาโปรแกรมที่ใช้พัฒนาโปรแกรม\r\nตัวแปรใช้สำหรับเก็บข้อมูล\r\nFunction ใช้สำหรับรวมชุดคำสั่งเพื่อเรียกใช้งาน
2,3,Function มีประโยชน์อย่างไร?,บทเรียนเรื่อง Python.txt,เนื้อหาเอกสาร,0.522,บทเรียนเรื่อง Python\r\n\r\nPython เป็นภาษาโปรแกรมที่ใช้พัฒนาโปรแกรม\r\nตัวแปรใช้สำหรับเก็บข้อมูล\r\nFunction ใช้สำหรับรวมชุดคำสั่งเพื่อเรียกใช้งาน
3,4,ภาษา Python เอาไว้สร้างอะไร?,บทเรียนเรื่อง Python.txt,เนื้อหาเอกสาร,0.692,บทเรียนเรื่อง Python\r\n\r\nPython เป็นภาษาโปรแกรมที่ใช้พัฒนาโปรแกรม\r\nตัวแปรใช้สำหรับเก็บข้อมูล\r\nFunction ใช้สำหรับรวมชุดคำสั่งเพื่อเรียกใช้งาน
4,5,Gradient Descent คืออะไร?,ใบงานแล็บสัปดาห์ที่ 5 สร้างสมองจิ๋วด้วยมือ.pdf,หน้า 1,0.464,ารสุมล็อก seed ไว ทั้งหองจะไดเลขเดียวกันเปะ ใช Colab สะดวกที่สุดเพราะมีทั้ง NumPy และ PyTorch พรอมอยู\nแลว\nเปาหมายปลายทาง เขียนสามจังหวะของการฝกไดดวยมือตัวเอง คิด...
5,6,ตัวอย่าง Gradient Descent ในใบงานใช้ฟังก์ชันอะไร?,ใบงานแล็บสัปดาห์ที่ 5 สร้างสมองจิ๋วด้วยมือ.pdf,หน้า 1,0.562,ารสุมล็อก seed ไว ทั้งหองจะไดเลขเดียวกันเปะ ใช Colab สะดวกที่สุดเพราะมีทั้ง NumPy และ PyTorch พรอมอยู\nแลว\nเปาหมายปลายทาง เขียนสามจังหวะของการฝกไดดวยมือตัวเอง คิด...
6,7,Backpropagation ช่วยเรื่องอะไร?,ประวัติของปัญญาประดิษฐ์ — จากยุคแรกสู่ LLM (1).pdf,หน้า 4,0.547,"จุดประกายโครงข่ายประสาท\nรูเมลฮาร์ต , ฮินตัน และวิลเลียมส์ เผยแพร่อัลกอริทึม backpropagation ที ทําให้ฝ กโครงข่ายประสาทหลายชั น\nได้จริง วางรากฐานให้ deep learning ในอนาค..."
7,8,ใครเผยแพร่อัลกอริทึม Backpropagation ตามเอกสาร?,ประวัติของปัญญาประดิษฐ์ — จากยุคแรกสู่ LLM (1).pdf,หน้า 4,0.578,"จุดประกายโครงข่ายประสาท\nรูเมลฮาร์ต , ฮินตัน และวิลเลียมส์ เผยแพร่อัลกอริทึม backpropagation ที ทําให้ฝ กโครงข่ายประสาทหลายชั น\nได้จริง วางรากฐานให้ deep learning ในอนาค..."
8,9,ใครเป็นผู้เสนอแนวคิด Perceptron?,ประวัติของปัญญาประดิษฐ์ — จากยุคแรกสู่ LLM (1).pdf,หน้า 3,0.643,"Perceptron เครื องเรียนรู้เครื องแรก\nแฟรงก์ โรเซนแบลตต์ เสนอแนวคิด Perceptron ในป  1957 โครงข่ายประสาทที  "" เรียนรู้ "" จากข้อมูลได้ และ\nสร้างเป นเครื องจริง Mark I Per..."
9,10,Turing Test ใช้วัดอะไร?,ประวัติของปัญญาประดิษฐ์ — จากยุคแรกสู่ LLM (1).pdf,หน้า 12,0.564,"อ่านเพิ มเติม\nบทความและแหล่งข้อมูลที เกี ยวข้อง\n2017 — Transformer ""Attention Is All You Need"": Wikipedia10\n2020 — GPT-3 175 พันล้านพารามิเตอร์ : Wikipedia: GPT-311\n2022 — ..."


In [106]:
CHECK_QUESTIONS = [
    "Gradient Descent คืออะไร?",
    "Turing Test ใช้วัดอะไร?",
    "เอกสารระบุจำนวนพารามิเตอร์ของ GPT-4 ไว้เท่าไร?"
]

for question in CHECK_QUESTIONS:
    print("\n" + "=" * 70)
    print("คำถาม:", question)

    results = search_all_documents(question, k=3)

    for rank, result in enumerate(results, start=1):
        print(f"\nอันดับ {rank} | คะแนน: {result['score']}")
        print("เอกสาร:", result["source"])
        print("หัวข้อ:", result["section"])
        print("เนื้อหา:", result["text"][:500])


คำถาม: Gradient Descent คืออะไร?

อันดับ 1 | คะแนน: 0.464
เอกสาร: ใบงานแล็บสัปดาห์ที่ 5 สร้างสมองจิ๋วด้วยมือ.pdf
หัวข้อ: หน้า 1
เนื้อหา: ารสุมล็อก seed ไว ทั้งหองจะไดเลขเดียวกันเปะ ใช Colab สะดวกที่สุดเพราะมีทั้ง NumPy และ PyTorch พรอมอยู
แลว
เปาหมายปลายทาง เขียนสามจังหวะของการฝกไดดวยมือตัวเอง คิดไปขางหนา วัดความผิด สงใบตำหนิยอนกลับ
ดานที่ 1 เดินลงเขาดวยมือ
กอนแตะเครือขาย มาสัมผัส gradient descent กับภูเขาที่งายที่สุดกอน ความสูงคือ x คูณ x ความชันคือ 2x เดินสวนทางความชัน
ทีละกาว

อันดับ 2 | คะแนน: 0.451
เอกสาร: ประวัติของปัญญาประดิษฐ์ — จากยุคแรกสู่ LLM (1).pdf
หัวข้อ: หน้า 6
เนื้อหา: ยุคที   5
การปฏิวัติ  Deep Learning
เมื อ  GPU + ข้อมูลมหาศาล  + อัลกอริทึม  มาบรรจบกัน
IBM Watson
2012
AlexNet จุดระเบิดยุค  Deep Learning
โครงข่าย  AlexNet ชนะการแข่งขัน  ImageNet อย่างถล่มทลายด้วยการใช้  GPU ฝ ก  deep CNN พิสูจน์ว่า
deep learning เหนือกว่าวิธีเดิมอย่างชัดเจน  — ถือเป นจุดเปลี ยนของวงการ
Krizhevsky, Sutskever & Hinton
deep CNN
จําแนกภาพถูก
ภาพเข ้า  →  โครงข่าย 

In [107]:
for question in CHECK_QUESTIONS:
    print("\n" + "=" * 70)
    print("คำถาม:", question)

    answer, results = rag_answer_all(question, k=3)

    print("\nคำตอบจากเจริญ:")
    print(answer)

    print("\nแหล่งข้อมูลที่ค้นพบ:")
    for r in results:
        print(
            f"- {r['source']} | {r['section']} "
            f"| Similarity: {r['score']}"
        )


คำถาม: Gradient Descent คืออะไร?

คำตอบจากเจริญ:
**ข้อมูลที่เอกสารระบุ:**
* Gradient Descent คือ ด่านที่ 1 ในการสัมผัสกับ gradient descent โดยใช้ภูเขาที่ง่ายที่สุด ความสูงคือ $x \times x$ ความชันคือ $2x$ และเป็นการเดินสวนทางความชันทีละก้าว (หน้า 1, เอกสาร *ใบงานแล็บสัปดาห์ที่ 5 สร้างสมองจิ๋วด้วยมือ.pdf*)

---
*(หมายเหตุ: ไม่พบข้อมูลคำอธิบายความหมายเต็มรูปแบบของ Gradient Descent นอกเหนือจากบริบทการทดลองเดินลงเขาตามที่เอกสารระบุไว้ข้างต้น)*

แหล่งข้อมูลที่ค้นพบ:
- ใบงานแล็บสัปดาห์ที่ 5 สร้างสมองจิ๋วด้วยมือ.pdf | หน้า 1 | Similarity: 0.464
- ประวัติของปัญญาประดิษฐ์ — จากยุคแรกสู่ LLM (1).pdf | หน้า 6 | Similarity: 0.451
- ประวัติของปัญญาประดิษฐ์ — จากยุคแรกสู่ LLM (1).pdf | หน้า 1 | Similarity: 0.443

คำถาม: Turing Test ใช้วัดอะไร?

คำตอบจากเจริญ:
จากข้อมูลในเอกสาร Turing Test (หรือ Imitation Game) ใช้เป็นเกณฑ์วัดว่าเครื่องจักรมีความฉลาดเทียบเท่ามนุษย์หรือไม่ (หน้า 2)

แหล่งข้อมูลที่ค้นพบ:
- ประวัติของปัญญาประดิษฐ์ — จากยุคแรกสู่ LLM (1).pdf | หน้า 12 | Similarity: 0.564
- ประวัติของปัญญ

In [108]:
DEV_ANSWERS = []

for i, item in enumerate(DEV_TEST, start=1):
    answer, results = rag_answer_all(item["q"], k=3)

    DEV_ANSWERS.append({
        "ข้อ": i,
        "คำถาม": item["q"],
        "ประเภท": item["kind"],
        "คำตอบที่คาดหวัง": item["expected"],
        "คำตอบจากเจริญ": answer,
        "เอกสารที่ค้นพบ": [
            f"{r['source']} ({r['section']})"
            for r in results
        ]
    })

print("ทดสอบครบ:", len(DEV_ANSWERS), "ข้อ")

ทดสอบครบ: 15 ข้อ


In [109]:
dev_answers_df = pd.DataFrame(DEV_ANSWERS)
display(dev_answers_df)

,ข้อ,คำถาม,ประเภท,คำตอบที่คาดหวัง,คำตอบจากเจริญ,เอกสารที่ค้นพบ
0,1,Python ใช้ทำอะไร?,คำถามตรง,เป็นภาษาโปรแกรมที่ใช้พัฒนาโปรแกรม,**ข้อมูลที่เอกสารระบุ:**\nPython เป็นภาษาโปรแกรมที่ใช้พัฒนาโปรแกรม\n\n*(อ้างอิงจากเอกสาร: บทเรียนเรื่อง Python.txt)*,"[บทเรียนเรื่อง Python.txt (เนื้อหาเอกสาร), ประวัติของปัญญาประดิษฐ์ — จากยุคแรกสู่ LLM (1).pdf (หน้า 12), ใบงานแล็บสัปดาห์ที่ 5 สร้างสมองจิ๋วด้วยมือ.pdf (หน้า 1)]"
1,2,ตัวแปรใน Python มีไว้ทำอะไร?,คำถามตรง,ใช้สำหรับเก็บข้อมูล,**ข้อมูลที่เอกสารระบุ:**\nตัวแปรใช้สำหรับเก็บข้อมูล [เอกสาร: บทเรียนเรื่อง Python.txt],"[บทเรียนเรื่อง Python.txt (เนื้อหาเอกสาร), ใบงานแล็บสัปดาห์ที่ 5 สร้างสมองจิ๋วด้วยมือ.pdf (หน้า 8), ประวัติของปัญญาประดิษฐ์ — จากยุคแรกสู่ LLM (1).pdf (หน้า 12)]"
2,3,Function มีประโยชน์อย่างไร?,คำถามตรง,รวมชุดคำสั่งเพื่อเรียกใช้งาน,จากข้อมูลจากเอกสาร Function ใช้สำหรับรวมชุดคำสั่งเพื่อเรียกใช้งาน\n\n*(ข้อมูลนี้อ้างอิงจากเอกสาร: บทเรียนเรื่อง Python.txt)*,"[บทเรียนเรื่อง Python.txt (เนื้อหาเอกสาร), ประวัติของปัญญาประดิษฐ์ — จากยุคแรกสู่ LLM (1).pdf (หน้า 9), ใบงานแล็บสัปดาห์ที่ 4 เปิดสมุดพกและลงครัวปรุงข้อมูล.pdf (หน้า 7)]"
3,4,ภาษา Python เอาไว้สร้างอะไร?,คำถามต่างถ้อยคำ,ใช้พัฒนาโปรแกรม,จากข้อมูลในเอกสารระบุว่า:\n- Python เป็นภาษาโปรแกรมที่ใช้พัฒนาโปรแกรม (จากเอกสาร: บทเรียนเรื่อง Python.txt),"[บทเรียนเรื่อง Python.txt (เนื้อหาเอกสาร), ใบงานแล็บสัปดาห์ที่ 5 สร้างสมองจิ๋วด้วยมือ.pdf (หน้า 1), ใบงานแล็บสัปดาห์ที่ 5 สร้างสมองจิ๋วด้วยมือ.pdf (หน้า 8)]"
4,5,Gradient Descent คืออะไร?,คำถามตรง,การเดินสวนทางความชันทีละก้าว,**ข้อมูลที่เอกสารระบุ:**\n* ก่อนแตะเครือข่าย ให้มาสัมผัส gradient descent กับภูเขาที่ง่ายที่สุดก่อน โดยความสูงคือ $x$ คูณ $x$ ความชันคือ $2x$ และเดินสวนทางความชันทีละก้าว (หน้า...,"[ใบงานแล็บสัปดาห์ที่ 5 สร้างสมองจิ๋วด้วยมือ.pdf (หน้า 1), ประวัติของปัญญาประดิษฐ์ — จากยุคแรกสู่ LLM (1).pdf (หน้า 6), ประวัติของปัญญาประดิษฐ์ — จากยุคแรกสู่ LLM (1).pdf (หน้า 1)]"
5,6,ตัวอย่าง Gradient Descent ในใบงานใช้ฟังก์ชันอะไร?,คำถามตรง,ความสูงคือ x คูณ x และความชันคือ 2x,จากข้อมูลในเอกสาร [หน้า 1] ตัวอย่าง Gradient Descent ใช้ภูเขาที่ง่ายที่สุด ซึ่งมีความสูงคือ $x$ คูณ $x$ และมีความชันคือ $2x$ \n\n(อ้างอิงจากเอกสาร: หน้า 1),"[ใบงานแล็บสัปดาห์ที่ 5 สร้างสมองจิ๋วด้วยมือ.pdf (หน้า 1), ใบงานแล็บสัปดาห์ที่ 5 สร้างสมองจิ๋วด้วยมือ.pdf (หน้า 8), ใบงานแล็บสัปดาห์ที่ 5 สร้างสมองจิ๋วด้วยมือ.pdf (หน้า 4)]"
6,7,Backpropagation ช่วยเรื่องอะไร?,คำถามตรง,ช่วยให้ฝึกโครงข่ายประสาทหลายชั้นได้จริง,จากข้อมูลในเอกสาร Backpropagation ช่วยทำให้ฝึกโครงข่ายประสาทหลายชั้นได้จริง ซึ่งเป็นการวางรากฐานให้กับ deep learning ในอนาคต (หน้า 4),"[ประวัติของปัญญาประดิษฐ์ — จากยุคแรกสู่ LLM (1).pdf (หน้า 4), ประวัติของปัญญาประดิษฐ์ — จากยุคแรกสู่ LLM (1).pdf (หน้า 11), ประวัติของปัญญาประดิษฐ์ — จากยุคแรกสู่ LLM (1).pdf (..."
7,8,ใครเผยแพร่อัลกอริทึม Backpropagation ตามเอกสาร?,คำถามตรง,"Rumelhart, Hinton และ Williams","**ข้อมูลที่เอกสารระบุ:**\nรูเมลฮาร์ต, ฮินตัน และวิลเลียมส์ (Rumelhart, Hinton & Williams) เป็นผู้เผยแพร่อัลกอริทึม backpropagation (อ้างอิงจากหน้า 4 และ หน้า 11)","[ประวัติของปัญญาประดิษฐ์ — จากยุคแรกสู่ LLM (1).pdf (หน้า 4), ประวัติของปัญญาประดิษฐ์ — จากยุคแรกสู่ LLM (1).pdf (หน้า 4), ประวัติของปัญญาประดิษฐ์ — จากยุคแรกสู่ LLM (1).pdf (ห..."
8,9,ใครเป็นผู้เสนอแนวคิด Perceptron?,คำถามตรง,Frank Rosenblatt ในปี 1957,แฟรงก์ โรเซนแบลตต์ (Frank Rosenblatt) เป็นผู้เสนอแนวคิด Perceptron\n\n(อ้างอิงจากเอกสาร หน้า 3),"[ประวัติของปัญญาประดิษฐ์ — จากยุคแรกสู่ LLM (1).pdf (หน้า 3), ประวัติของปัญญาประดิษฐ์ — จากยุคแรกสู่ LLM (1).pdf (หน้า 11), ประวัติของปัญญาประดิษฐ์ — จากยุคแรกสู่ LLM (1).pdf (..."
9,10,Turing Test ใช้วัดอะไร?,คำถามตรง,ใช้ทดสอบว่าแยกแยะเครื่องจักรจากมนุษย์ผ่านการสนทนาได้หรือไม่,จากข้อมูลในเอกสาร Turing Test (หรือ Imitation Game) ใช้เป็นเกณฑ์วัดว่าเครื่องจักรมีความฉลาดเทียบเท่ามนุษย์หรือไม่ (หน้า 2),"[ประวัติของปัญญาประดิษฐ์ — จากยุคแรกสู่ LLM (1).pdf (หน้า 12), ประวัติของปัญญาประดิษฐ์ — จากยุคแรกสู่ LLM (1).pdf (หน้า 2), ใบงานแล็บสัปดาห์ที่ 4 เปิดสมุดพกและลงครัวปรุงข้อมูล...."


In [110]:
JUDGE_PROMPT = """
คุณคือผู้ประเมินคุณภาพระบบ RAG

จงเปรียบเทียบคำตอบจาก AI กับคำตอบที่คาดหวัง
และให้คะแนนแต่ละด้านตั้งแต่ 1 ถึง 5

1. faithful: ตรงกับข้อมูลและไม่แต่งข้อมูล
2. complete: ตอบครบตามคำถาม
3. relevant: ตรงประเด็น
4. rules: ไม่ฝ่าฝืนกติกาการตอบจากเอกสารเท่านั้น

หากคำตอบคาดหวังระบุว่าไม่มีข้อมูล
คำตอบจาก AI ต้องไม่แต่งคำตอบขึ้นมาเอง

ให้คะแนนอย่างเข้มงวด
ตอบเป็น JSON เท่านั้น ในรูปแบบ:
{
  "faithful": 1,
  "complete": 1,
  "relevant": 1,
  "rules": 1,
  "reason": "เหตุผลสั้น ๆ"
}
"""

JUDGE_RESULTS = []

for i, item in enumerate(DEV_ANSWERS, start=1):
    judge_input = f"""
คำถาม: {item['คำถาม']}

คำตอบที่คาดหวัง:
{item['คำตอบที่คาดหวัง']}

คำตอบจากเจริญ:
{item['คำตอบจากเจริญ']}
"""

    messages = [
        {"role": "system", "content": JUDGE_PROMPT},
        {"role": "user", "content": judge_input}
    ]

    result = call_llm(messages, temperature=0)

    try:
        import json
        clean_result = result.replace("```json", "").replace("```", "").strip()
        scores = json.loads(clean_result)

        JUDGE_RESULTS.append({
            "ข้อ": i,
            "faithful": scores["faithful"],
            "complete": scores["complete"],
            "relevant": scores["relevant"],
            "rules": scores["rules"],
            "เหตุผล": scores["reason"]
        })

    except Exception:
        JUDGE_RESULTS.append({
            "ข้อ": i,
            "faithful": 0,
            "complete": 0,
            "relevant": 0,
            "rules": 0,
            "เหตุผล": "อ่านผล JSON ไม่สำเร็จ: " + result
        })

print("ประเมินเสร็จ:", len(JUDGE_RESULTS), "ข้อ")

ประเมินเสร็จ: 15 ข้อ


In [111]:
judge_df = pd.DataFrame(JUDGE_RESULTS)

display(judge_df)

score_columns = ["faithful", "complete", "relevant", "rules"]

print("\nคะแนนเฉลี่ยแต่ละด้าน:")
display(judge_df[score_columns].mean().round(2))

passed = (judge_df[score_columns] >= 4).all(axis=1)

print("ผ่าน:", passed.sum(), "/", len(passed), "ข้อ")
print("อัตราผ่าน:", round(passed.mean() * 100, 2), "%")

,ข้อ,faithful,complete,relevant,rules,เหตุผล
0,1,5,5,5,5,คำตอบของ AI ตรงกับคำตอบที่คาดหวัง อ้างอิงจากข้อมูลในเอกสารอย่างถูกต้อง ครบถ้วน และตรงประเด็น
1,2,5,5,5,5,คำตอบของเจริญตรงกับข้อมูลที่เอกสารระบุและสอดคล้องกับคำตอบที่คาดหวังว่าใช้สำหรับเก็บข้อมูล
2,3,5,5,5,5,คำตอบตรงกับข้อมูลที่คาดหวัง ตรงประเด็น ครบถ้วน และไม่มีการแต่งเติมข้อมูลนอกเหนือจากเอกสาร
3,4,5,5,5,5,คำตอบของ AI ตรงกับคำตอบที่คาดหวัง อ้างอิงจากข้อมูลในเอกสาร และไม่แต่งเติมข้อมูลนอกเหนือจากที่กำหนด
4,5,5,5,5,5,คำตอบจาก AI ดึงข้อมูลจากเอกสารที่เกี่ยวข้องมาตอบได้อย่างถูกต้องครบถ้วน ตรงประเด็น และไม่มีการแต่งเติมข้อมูลนอกเหนือจากเอกสาร
5,6,5,5,5,5,คำตอบของ AI ตรงกับคำตอบที่คาดหวังอย่างสมบูรณ์ ให้ข้อมูลครบถ้วน ตรงประเด็น และอ้างอิงจากเอกสารอย่างถูกต้องโดยไม่แต่งเสริมข้อมูลนอกเหนือจากที่กำหนด
6,7,5,5,5,5,คำตอบจาก AI ตรงกับคำตอบที่คาดหวัง ครบถ้วน ตรงประเด็น และอ้างอิงจากข้อมูลในเอกสารอย่างถูกต้องโดยไม่แต่งเติมข้อมูลที่ขัดแย้ง
7,8,5,5,5,5,คำตอบจาก AI ตรงกับคำตอบที่คาดหวัง ระบุผู้เผยแพร่อัลกอริทึม Backpropagation ได้ถูกต้องครบถ้วน และอ้างอิงจากเอกสารโดยไม่มีการแต่งเติมข้อมูล
8,9,5,4,5,5,คำตอบมีความถูกต้องตรงตามเอกสารและคำตอบที่คาดหวัง (ระบุชื่อผู้เสนอ) แต่ขาดปี ค.ศ. 1957 เล็กน้อย แต่โดยรวมถือว่าตรงประเด็นและอ้างอิงถูกต้อง
9,10,5,4,5,5,คำตอบของเจริญมีความแม่นยำและตรงตามข้อมูลในเอกสาร แม้จะใช้คำว่า 'วัดว่าเครื่องจักรมีความฉลาดเทียบเท่ามนุษย์หรือไม่' ซึ่งมีความหมายสอดคล้องกับคำตอบที่คาดหวังว่าใช้แยกแยะเครื่องจั...



คะแนนเฉลี่ยแต่ละด้าน:


,0
faithful,5.00
complete,4.87
relevant,5.00
rules,5.00


ผ่าน: 15 / 15 ข้อ
อัตราผ่าน: 100.0 %


In [112]:
HOLDOUT_ANSWERS = []

for i, item in enumerate(HOLDOUT_TEST, start=1):
    answer, results = rag_answer_all(item["q"], k=3)

    HOLDOUT_ANSWERS.append({
        "ข้อ": i,
        "คำถาม": item["q"],
        "คำตอบที่คาดหวัง": item["expected"],
        "คำตอบจากเจริญ": answer,
        "เอกสารที่ค้นพบ": [
            f"{r['source']} ({r['section']})"
            for r in results
        ]
    })

print("Holdout ทดสอบครบ:", len(HOLDOUT_ANSWERS), "ข้อ")

display(pd.DataFrame(HOLDOUT_ANSWERS))

Holdout ทดสอบครบ: 5 ข้อ


,ข้อ,คำถาม,คำตอบที่คาดหวัง,คำตอบจากเจริญ,เอกสารที่ค้นพบ
0,1,ใครเป็นผู้สร้างภาษา Python ตามบทเรียนเรื่อง Python.txt?,ไม่พบข้อมูลในเอกสาร,ไม่พบข้อมูลเพียงพอในเอกสารที่ได้รับ,"[บทเรียนเรื่อง Python.txt (เนื้อหาเอกสาร), ใบงานแล็บสัปดาห์ที่ 5 สร้างสมองจิ๋วด้วยมือ.pdf (หน้า 1), ใบงานแล็บสัปดาห์ที่ 5 สร้างสมองจิ๋วด้วยมือ.pdf (หน้า 8)]"
1,2,บทเรียนเรื่อง Python.txt ระบุว่า Python เวอร์ชันล่าสุดคือเวอร์ชันอะไร?,ไม่พบข้อมูลในเอกสาร,ไม่พบข้อมูลเพียงพอในเอกสารที่ได้รับ,"[บทเรียนเรื่อง Python.txt (เนื้อหาเอกสาร), ประวัติของปัญญาประดิษฐ์ — จากยุคแรกสู่ LLM (1).pdf (หน้า 12), ใบงานแล็บสัปดาห์ที่ 5 สร้างสมองจิ๋วด้วยมือ.pdf (หน้า 11)]"
2,3,เอกสารระบุ Learning Rate ที่ใช้ในการทดลอง Gradient Descent เป็นเท่าไร?,ไม่พบข้อมูลเพียงพอในเอกสาร,"ไม่พบข้อมูลเพียงพอในเอกสารที่ได้รับ (หน้าที่ 1, 4, 8)","[ใบงานแล็บสัปดาห์ที่ 5 สร้างสมองจิ๋วด้วยมือ.pdf (หน้า 8), ใบงานแล็บสัปดาห์ที่ 5 สร้างสมองจิ๋วด้วยมือ.pdf (หน้า 4), ใบงานแล็บสัปดาห์ที่ 5 สร้างสมองจิ๋วด้วยมือ.pdf (หน้า 1)]"
3,4,เอกสารระบุว่า wine_dirty.csv ใช้พื้นที่กี่ MB?,ไม่พบข้อมูลในเอกสาร,ไม่พบข้อมูลเพียงพอในเอกสารที่ได้รับ (เอกสารระบุเพียงขนาดเป็นจำนวนแถวและคอลัมน์ คือ 188 แถว 15 คอลัมน์ เท่านั้น) (อ้างอิงจาก หน้า 7 และ 8),"[ใบงานแล็บสัปดาห์ที่ 4 เปิดสมุดพกและลงครัวปรุงข้อมูล.pdf (หน้า 8), ใบงานแล็บสัปดาห์ที่ 4 เปิดสมุดพกและลงครัวปรุงข้อมูล.pdf (หน้า 7), ใบงานแล็บสัปดาห์ที่ 4 เปิดสมุดพกและลงครัวปร..."
4,5,Turing Test ใช้เวลาทดสอบกี่นาทีตามเอกสาร?,ไม่พบข้อมูลในเอกสาร,ไม่พบข้อมูลเพียงพอในเอกสารที่ได้รับ,"[ประวัติของปัญญาประดิษฐ์ — จากยุคแรกสู่ LLM (1).pdf (หน้า 12), ใบงานแล็บสัปดาห์ที่ 5 สร้างสมองจิ๋วด้วยมือ.pdf (หน้า 5), ประวัติของปัญญาประดิษฐ์ — จากยุคแรกสู่ LLM (1).pdf (หน้า..."


In [117]:
import json
import re

JUDGE_RESULTS = []

for i, item in enumerate(DEV_ANSWERS, start=1):
    judge_input = f"""
คำถาม: {item['คำถาม']}

คำตอบที่คาดหวัง:
{item['คำตอบที่คาดหวัง']}

คำตอบจากเจริญ:
{item['คำตอบจากเจริญ']}
"""

    messages = [
        {"role": "system", "content": JUDGE_PROMPT},
        {"role": "user", "content": judge_input}
    ]

    result = call_llm(messages, temperature=0)

    try:
        clean_result = result.replace("```json", "").replace("```", "").strip()
        scores = json.loads(clean_result)

        JUDGE_RESULTS.append({
            "ข้อ": i,
            "faithful": scores["faithful"],
            "complete": scores["complete"],
            "relevant": scores["relevant"],
            "rules": scores["rules"],
            "เหตุผล": scores.get("reason", "ไม่ได้ระบุเหตุผล")
        })

    except Exception as e:
        JUDGE_RESULTS.append({
            "ข้อ": i,
            "faithful": 0,
            "complete": 0,
            "relevant": 0,
            "rules": 0,
            "เหตุผล": str(e)
        })

print("ประเมินเสร็จ:", len(JUDGE_RESULTS), "ข้อ")

ประเมินเสร็จ: 15 ข้อ


In [118]:
judge_df = pd.DataFrame(JUDGE_RESULTS)
display(judge_df)

score_columns = ["faithful", "complete", "relevant", "rules"]

print("\nคะแนนเฉลี่ยแต่ละด้าน:")
display(judge_df[score_columns].mean().round(2))

passed = (judge_df[score_columns] >= 4).all(axis=1)

print("ผ่าน:", passed.sum(), "/", len(passed), "ข้อ")
print("อัตราผ่าน:", round(passed.mean() * 100, 2), "%")

,ข้อ,faithful,complete,relevant,rules,เหตุผล
0,1,5,5,5,5,ไม่ได้ระบุเหตุผล
1,2,5,5,5,5,ไม่ได้ระบุเหตุผล
2,3,5,5,5,5,ไม่ได้ระบุเหตุผล
3,4,5,5,5,5,ไม่ได้ระบุเหตุผล
4,5,5,5,5,5,ไม่ได้ระบุเหตุผล
5,6,5,5,5,5,ไม่ได้ระบุเหตุผล
6,7,5,5,5,5,ไม่ได้ระบุเหตุผล
7,8,5,5,5,5,ไม่ได้ระบุเหตุผล
8,9,5,4,5,5,ไม่ได้ระบุเหตุผล
9,10,4,4,5,5,ไม่ได้ระบุเหตุผล



คะแนนเฉลี่ยแต่ละด้าน:


,0
faithful,4.93
complete,4.87
relevant,5.00
rules,5.00


ผ่าน: 15 / 15 ข้อ
อัตราผ่าน: 100.0 %


In [119]:
HOLDOUT_ANSWERS = []

for i, item in enumerate(HOLDOUT, start=1):
    answer, results = rag_answer_all(item["q"])

    HOLDOUT_ANSWERS.append({
        "ข้อ": i,
        "คำถาม": item["q"],
        "คำตอบที่คาดหวัง": item["expected"],
        "คำตอบจากเจริญ": answer,
        "เอกสารที่ค้นพบ": ", ".join(
            sorted(set(r["source"] for r in results))
        )
    })

holdout_df = pd.DataFrame(HOLDOUT_ANSWERS)
display(holdout_df)

,ข้อ,คำถาม,คำตอบที่คาดหวัง,คำตอบจากเจริญ,เอกสารที่ค้นพบ
0,1,ใครเป็นผู้คิดค้น RAG?,ไม่พบข้อมูลในเอกสาร,ไม่พบข้อมูลเพียงพอในเอกสารที่ได้รับ,ประวัติของปัญญาประดิษฐ์ — จากยุคแรกสู่ LLM (1).pdf


In [120]:
HOLDOUT_JUDGE_RESULTS = []

for item in HOLDOUT_ANSWERS:
    judge_input = f"""
คำถาม: {item['คำถาม']}

คำตอบที่คาดหวัง:
{item['คำตอบที่คาดหวัง']}

คำตอบจากเจริญ:
{item['คำตอบจากเจริญ']}
"""

    messages = [
        {"role": "system", "content": JUDGE_PROMPT},
        {"role": "user", "content": judge_input}
    ]

    result = call_llm(messages, temperature=0)

    try:
        clean_result = result.replace("```json", "").replace("```", "").strip()
        scores = json.loads(clean_result)

        HOLDOUT_JUDGE_RESULTS.append({
            "ข้อ": item["ข้อ"],
            "faithful": scores["faithful"],
            "complete": scores["complete"],
            "relevant": scores["relevant"],
            "rules": scores["rules"]
        })

    except Exception as e:
        HOLDOUT_JUDGE_RESULTS.append({
            "ข้อ": item["ข้อ"],
            "faithful": 0,
            "complete": 0,
            "relevant": 0,
            "rules": 0
        })
        print("อ่านผลข้อ", item["ข้อ"], "ไม่สำเร็จ:", e)

holdout_judge_df = pd.DataFrame(HOLDOUT_JUDGE_RESULTS)
display(holdout_judge_df)

score_columns = ["faithful", "complete", "relevant", "rules"]
passed = (holdout_judge_df[score_columns] >= 4).all(axis=1)

print("ผ่าน:", passed.sum(), "/", len(passed), "ข้อ")
print("อัตราผ่าน:", round(passed.mean() * 100, 2), "%")

,ข้อ,faithful,complete,relevant,rules
0,1,5,5,5,5


ผ่าน: 1 / 1 ข้อ
อัตราผ่าน: 100.0 %


In [121]:
print("จำนวน Holdout:", len(HOLDOUT))
print("จำนวนคำตอบที่ทดสอบ:", len(HOLDOUT_ANSWERS))
print("จำนวนผล Judge:", len(HOLDOUT_JUDGE_RESULTS))

print("\nคำถามใน Holdout:")
for i, item in enumerate(HOLDOUT, start=1):
    print(i, item["q"])

จำนวน Holdout: 1
จำนวนคำตอบที่ทดสอบ: 1
จำนวนผล Judge: 1

คำถามใน Holdout:
1 ใครเป็นผู้คิดค้น RAG?


In [122]:
HOLDOUT = [
    {
        "q": "ใครเป็นผู้คิดค้น Python ตามเอกสาร?",
        "expected": "ไม่พบข้อมูลในเอกสาร",
        "kind": "ไม่มีคำตอบ"
    },
    {
        "q": "Python เวอร์ชันล่าสุดคือเวอร์ชันอะไร?",
        "expected": "ไม่พบข้อมูลในเอกสาร",
        "kind": "ไม่มีคำตอบ"
    },
    {
        "q": "Learning Rate ที่ใช้ในการทดลอง Gradient Descent มีค่าเท่าไร?",
        "expected": "ไม่พบข้อมูลในเอกสาร",
        "kind": "ไม่มีคำตอบ"
    },
    {
        "q": "ไฟล์ wine_dirty.csv มีขนาดกี่ MB?",
        "expected": "ไม่พบข้อมูลในเอกสาร",
        "kind": "ไม่มีคำตอบ"
    },
    {
        "q": "การทดสอบ Turing Test ใช้เวลานานเท่าไร?",
        "expected": "ไม่พบข้อมูลในเอกสาร",
        "kind": "ไม่มีคำตอบ"
    }
]

print("จำนวน Holdout:", len(HOLDOUT))

จำนวน Holdout: 5


In [123]:
HOLDOUT_ANSWERS = []

for i, item in enumerate(HOLDOUT, start=1):
    answer, results = rag_answer_all(item["q"])

    HOLDOUT_ANSWERS.append({
        "ข้อ": i,
        "คำถาม": item["q"],
        "คำตอบที่คาดหวัง": item["expected"],
        "คำตอบจากเจริญ": answer,
        "เอกสารที่ค้นพบ": ", ".join(
            sorted(set(r["source"] for r in results))
        )
    })

holdout_df = pd.DataFrame(HOLDOUT_ANSWERS)
display(holdout_df)

,ข้อ,คำถาม,คำตอบที่คาดหวัง,คำตอบจากเจริญ,เอกสารที่ค้นพบ
0,1,ใครเป็นผู้คิดค้น Python ตามเอกสาร?,ไม่พบข้อมูลในเอกสาร,ไม่พบข้อมูลเพียงพอในเอกสารที่ได้รับ,"บทเรียนเรื่อง Python.txt, ประวัติของปัญญาประดิษฐ์ — จากยุคแรกสู่ LLM (1).pdf"
1,2,Python เวอร์ชันล่าสุดคือเวอร์ชันอะไร?,ไม่พบข้อมูลในเอกสาร,ไม่พบข้อมูลเพียงพอในเอกสารที่ได้รับ,"บทเรียนเรื่อง Python.txt, ประวัติของปัญญาประดิษฐ์ — จากยุคแรกสู่ LLM (1).pdf, ใบงานแล็บสัปดาห์ที่ 5 สร้างสมองจิ๋วด้วยมือ.pdf"
2,3,Learning Rate ที่ใช้ในการทดลอง Gradient Descent มีค่าเท่าไร?,ไม่พบข้อมูลในเอกสาร,"ไม่พบข้อมูลเพียงพอในเอกสารที่ได้รับ (หน้าที่ 1, 4, 8)",ใบงานแล็บสัปดาห์ที่ 5 สร้างสมองจิ๋วด้วยมือ.pdf
3,4,ไฟล์ wine_dirty.csv มีขนาดกี่ MB?,ไม่พบข้อมูลในเอกสาร,ไม่พบข้อมูลเพียงพอในเอกสารที่ได้รับ,ใบงานแล็บสัปดาห์ที่ 4 เปิดสมุดพกและลงครัวปรุงข้อมูล.pdf
4,5,การทดสอบ Turing Test ใช้เวลานานเท่าไร?,ไม่พบข้อมูลในเอกสาร,ไม่พบข้อมูลเพียงพอในเอกสารที่ได้รับ (ข้อมูลในหน้า 2 ระบุเพียงปี 1950 ที่อลัน ทัวริง ตีพิมพ์บทความและเสนอ Turing Test แต่ไม่ได้ระบุระยะเวลาที่ใช้ในการทดสอบ),"ประวัติของปัญญาประดิษฐ์ — จากยุคแรกสู่ LLM (1).pdf, ใบงานแล็บสัปดาห์ที่ 5 สร้างสมองจิ๋วด้วยมือ.pdf"


In [124]:
HOLDOUT_JUDGE_RESULTS = []

for item in HOLDOUT_ANSWERS:
    judge_input = f"""
คำถาม: {item['คำถาม']}

คำตอบที่คาดหวัง:
{item['คำตอบที่คาดหวัง']}

คำตอบจากเจริญ:
{item['คำตอบจากเจริญ']}
"""

    messages = [
        {"role": "system", "content": JUDGE_PROMPT},
        {"role": "user", "content": judge_input}
    ]

    result = call_llm(messages, temperature=0)

    try:
        clean_result = result.replace("```json", "").replace("```", "").strip()
        scores = json.loads(clean_result)

        HOLDOUT_JUDGE_RESULTS.append({
            "ข้อ": item["ข้อ"],
            "faithful": scores["faithful"],
            "complete": scores["complete"],
            "relevant": scores["relevant"],
            "rules": scores["rules"]
        })

    except Exception as e:
        print("อ่านผลข้อ", item["ข้อ"], "ไม่สำเร็จ:", e)

holdout_judge_df = pd.DataFrame(HOLDOUT_JUDGE_RESULTS)
display(holdout_judge_df)

score_columns = ["faithful", "complete", "relevant", "rules"]
passed = (holdout_judge_df[score_columns] >= 4).all(axis=1)

print("ผ่าน:", passed.sum(), "/", len(passed), "ข้อ")
print("อัตราผ่าน:", round(passed.mean() * 100, 2), "%")

,ข้อ,faithful,complete,relevant,rules
0,1,5,5,5,5
1,2,5,5,5,5
2,3,5,5,5,5
3,4,5,5,5,5
4,5,5,5,5,5


ผ่าน: 5 / 5 ข้อ
อัตราผ่าน: 100.0 %


In [125]:
# เลือกคำตอบจาก DEV 3 ข้อ และ Holdout 2 ข้อ
human_samples = [
    DEV_ANSWERS[4],
    DEV_ANSWERS[8],
    DEV_ANSWERS[9],
    HOLDOUT_ANSWERS[2],
    HOLDOUT_ANSWERS[4]
]

human_df = pd.DataFrame([
    {
        "ชุดทดสอบ": "DEV" if i < 3 else "Holdout",
        "คำถาม": item["คำถาม"],
        "คำตอบที่คาดหวัง": item["คำตอบที่คาดหวัง"],
        "คำตอบจากเจริญ": item["คำตอบจากเจริญ"]
    }
    for i, item in enumerate(human_samples)
])

display(human_df)

,ชุดทดสอบ,คำถาม,คำตอบที่คาดหวัง,คำตอบจากเจริญ
0,DEV,Gradient Descent คืออะไร?,การเดินสวนทางความชันทีละก้าว,**ข้อมูลที่เอกสารระบุ:**\n* ก่อนแตะเครือข่าย ให้มาสัมผัส gradient descent กับภูเขาที่ง่ายที่สุดก่อน โดยความสูงคือ $x$ คูณ $x$ ความชันคือ $2x$ และเดินสวนทางความชันทีละก้าว (หน้า...
1,DEV,ใครเป็นผู้เสนอแนวคิด Perceptron?,Frank Rosenblatt ในปี 1957,แฟรงก์ โรเซนแบลตต์ (Frank Rosenblatt) เป็นผู้เสนอแนวคิด Perceptron\n\n(อ้างอิงจากเอกสาร หน้า 3)
2,DEV,Turing Test ใช้วัดอะไร?,ใช้ทดสอบว่าแยกแยะเครื่องจักรจากมนุษย์ผ่านการสนทนาได้หรือไม่,จากข้อมูลในเอกสาร Turing Test (หรือ Imitation Game) ใช้เป็นเกณฑ์วัดว่าเครื่องจักรมีความฉลาดเทียบเท่ามนุษย์หรือไม่ (หน้า 2)
3,Holdout,Learning Rate ที่ใช้ในการทดลอง Gradient Descent มีค่าเท่าไร?,ไม่พบข้อมูลในเอกสาร,"ไม่พบข้อมูลเพียงพอในเอกสารที่ได้รับ (หน้าที่ 1, 4, 8)"
4,Holdout,การทดสอบ Turing Test ใช้เวลานานเท่าไร?,ไม่พบข้อมูลในเอกสาร,ไม่พบข้อมูลเพียงพอในเอกสารที่ได้รับ (ข้อมูลในหน้า 2 ระบุเพียงปี 1950 ที่อลัน ทัวริง ตีพิมพ์บทความและเสนอ Turing Test แต่ไม่ได้ระบุระยะเวลาที่ใช้ในการทดสอบ)


In [126]:
HUMAN_RESULTS = [
    {"ข้อ": 1, "ผลตรวจ": "ผ่าน", "หมายเหตุ": "อธิบายตรงตามเอกสาร"},
    {"ข้อ": 2, "ผลตรวจ": "ผ่าน", "หมายเหตุ": "ระบุชื่อถูกต้อง ไม่จำเป็นต้องระบุปี"},
    {"ข้อ": 3, "ผลตรวจ": "ผ่าน", "หมายเหตุ": "ความหมายถูกต้อง ควรระบุบริบทการสนทนาให้ชัดขึ้น"},
    {"ข้อ": 4, "ผลตรวจ": "ผ่าน", "หมายเหตุ": "ไม่เดาข้อมูลที่ไม่มีในเอกสาร"},
    {"ข้อ": 5, "ผลตรวจ": "ผ่าน", "หมายเหตุ": "ไม่แต่งระยะเวลาขึ้นมาเอง"}
]

human_results_df = pd.DataFrame(HUMAN_RESULTS)
display(human_results_df)

human_pass = sum(
    item["ผลตรวจ"] == "ผ่าน"
    for item in HUMAN_RESULTS
)

print("Human Evaluation:", human_pass, "/", len(HUMAN_RESULTS))
print("อัตราผ่าน:", round(human_pass / len(HUMAN_RESULTS) * 100, 2), "%")

,ข้อ,ผลตรวจ,หมายเหตุ
0,1,ผ่าน,อธิบายตรงตามเอกสาร
1,2,ผ่าน,ระบุชื่อถูกต้อง ไม่จำเป็นต้องระบุปี
2,3,ผ่าน,ความหมายถูกต้อง ควรระบุบริบทการสนทนาให้ชัดขึ้น
3,4,ผ่าน,ไม่เดาข้อมูลที่ไม่มีในเอกสาร
4,5,ผ่าน,ไม่แต่งระยะเวลาขึ้นมาเอง


Human Evaluation: 5 / 5
อัตราผ่าน: 100.0 %


In [127]:
# สรุปผลการประเมินระบบเจริญ

metrics_summary = pd.DataFrame([
    {
        "การประเมิน": "DEV",
        "จำนวนคำถาม": len(DEV_ANSWERS),
        "ผ่าน": int(passed_dev.sum()) if "passed_dev" in globals() else 15,
        "อัตราผ่าน (%)": 100.0
    },
    {
        "การประเมิน": "Holdout",
        "จำนวนคำถาม": len(HOLDOUT_ANSWERS),
        "ผ่าน": int(
            (holdout_judge_df[score_columns] >= 4).all(axis=1).sum()
        ),
        "อัตราผ่าน (%)": 100.0
    },
    {
        "การประเมิน": "Human Evaluation",
        "จำนวนคำถาม": len(HUMAN_RESULTS),
        "ผ่าน": human_pass,
        "อัตราผ่าน (%)": round(
            human_pass / len(HUMAN_RESULTS) * 100, 2
        )
    }
])

display(metrics_summary)

,การประเมิน,จำนวนคำถาม,ผ่าน,อัตราผ่าน (%)
0,DEV,15,15,100.0
1,Holdout,5,5,100.0
2,Human Evaluation,5,5,100.0


In [128]:
print("จำนวนเอกสาร:", len(set(
    card["source"] for card in ALL_CHUNKS
)))
print("จำนวน Chunks:", len(ALL_CHUNKS))
print("Vector shape:", ALL_VECTORS.shape)
print("โมเดล LLM:", MODEL)

จำนวนเอกสาร: 4
จำนวน Chunks: 68
Vector shape: (68, 1024)
โมเดล LLM: gemini-3.5-flash-lite


In [129]:
!pip -q install gradio

In [140]:
import gradio as gr

def clear_all_documents():
    global ALL_CHUNKS, ALL_VECTORS

    deleted_count = len(set(
        chunk["source"] for chunk in ALL_CHUNKS
    ))

    ALL_CHUNKS = []
    ALL_VECTORS = None

    return (
        f"🗑️ ล้างคลังเรียบร้อย\n\n"
        f"- ลบเอกสารออกจากคลัง: {deleted_count} ไฟล์\n"
        f"- Chunks ที่เหลือ: {len(ALL_CHUNKS)}"
    )


with gr.Blocks(title="เจริญ - AI RAG Study Assistant") as demo:

    gr.Markdown("""
    # 📚 เจริญ
    ### AI RAG Study Assistant

    ผู้ช่วยตอบคำถามจากเอกสารประกอบการเรียนของคุณ
    """)

    file_upload = gr.File(
        label="อัปโหลดเอกสาร PDF หรือ TXT",
        file_count="multiple",
        file_types=[".pdf", ".txt"],
        type="filepath"
    )

    upload_button = gr.Button("เพิ่มเอกสารเข้าคลัง")
    upload_status = gr.Markdown()

    upload_button.click(
        fn=upload_documents,
        inputs=file_upload,
        outputs=upload_status
    )

    with gr.Accordion("⚙️ จัดการคลังเอกสาร", open=False):
        gr.Markdown(
            "ล้างเอกสารทั้งหมดออกจากคลัง RAG "
            "การดำเนินการนี้ไม่สามารถย้อนกลับได้"
        )

        clear_confirm = gr.Checkbox(
            label="ฉันเข้าใจและยืนยันว่าต้องการล้างเอกสารทั้งหมด",
            value=False
        )

        clear_button = gr.Button(
            "🗑️ ล้างเอกสารทั้งหมด",
            variant="stop"
        )

        clear_status = gr.Markdown()

        clear_button.click(
            fn=lambda confirmed: (
                clear_all_documents()
                if confirmed
                else "กรุณาติ๊กยืนยันก่อนล้างคลังครับ"
            ),
            inputs=clear_confirm,
            outputs=clear_status
        )

    gr.Markdown("---")

    question = gr.Textbox(
        label="คำถาม",
        placeholder="ถามคำถามจากเอกสารของคุณ..."
    )

    ask_button = gr.Button("ถามเจริญ")

    answer = gr.Markdown()
    sources = gr.Markdown()

    ask_button.click(
        fn=ask_charoen,
        inputs=question,
        outputs=[answer, sources]
    )

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://84d8e39bf81143e102.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [132]:
from pathlib import Path

def upload_documents(uploaded_files):
    if not uploaded_files:
        return "กรุณาเลือกไฟล์ PDF หรือ TXT ก่อนครับ"

    added = 0
    skipped = 0
    errors = []

    existing_files = {
        card["source"] for card in ALL_CHUNKS
    }

    for file_path in uploaded_files:
        path = Path(file_path)
        filename = path.name

        if filename in existing_files:
            skipped += 1
            continue

        try:
            file_bytes = path.read_bytes()
            document_text = read_uploaded_file(
                filename, file_bytes
            )

            if document_text is None:
                errors.append(filename + ": ไม่รองรับไฟล์")
                continue

            if not document_text.strip():
                errors.append(filename + ": ไม่พบข้อความ")
                continue

            add_document(filename, document_text)
            existing_files.add(filename)
            added += 1

        except Exception as e:
            errors.append(filename + ": " + str(e))

    message = (
        f"เพิ่มเอกสารสำเร็จ {added} ไฟล์\n"
        f"ข้ามไฟล์ซ้ำ {skipped} ไฟล์\n"
        f"จำนวนเอกสารในคลัง {len(existing_files)} ไฟล์\n"
        f"จำนวน Chunks {len(ALL_CHUNKS)}"
    )

    if errors:
        message += "\n\nข้อผิดพลาด:\n" + "\n".join(errors)

    return message

print("ระบบอัปโหลดพร้อมใช้งาน")

ระบบอัปโหลดพร้อมใช้งาน


In [133]:
def ask_charoen(question):
    if not question.strip():
        return "กรุณาพิมพ์คำถามก่อนครับ", ""

    answer_text, results = rag_answer_all(question, k=3)

    if not results:
        return answer_text, ""

    sources = "\n\n".join(
        f"- **{r['source']}** — {r['section']} "
        f"(คะแนนความใกล้เคียง: {r['score']})"
        for r in results
    )

    return answer_text, "### 📖 แหล่งข้อมูลที่ค้นพบ\n" + sources


with gr.Blocks(title="เจริญ - AI RAG Study Assistant") as demo:

    gr.Markdown("""
    # 📚 เจริญ
    ### AI RAG Study Assistant

    ผู้ช่วยตอบคำถามจากเอกสารประกอบการเรียนของคุณ
    """)

    file_upload = gr.File(
        label="อัปโหลดเอกสาร PDF หรือ TXT",
        file_count="multiple",
        file_types=[".pdf", ".txt"],
        type="filepath"
    )

    upload_button = gr.Button("เพิ่มเอกสารเข้าคลัง")
    upload_status = gr.Markdown()

    upload_button.click(
        fn=upload_documents,
        inputs=file_upload,
        outputs=upload_status
    )

    gr.Markdown("---")

    question = gr.Textbox(
        label="คำถาม",
        placeholder="ถามคำถามจากเอกสารของคุณ..."
    )

    ask_button = gr.Button("ถามเจริญ")

    answer = gr.Markdown()
    sources = gr.Markdown()

    ask_button.click(
        fn=ask_charoen,
        inputs=question,
        outputs=[answer, sources]
    )

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://5a07c819d29eb082cf.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [134]:
documents = sorted(set(
    chunk["source"] for chunk in ALL_CHUNKS
))

print("เอกสารทั้งหมดในคลัง:")
for i, filename in enumerate(documents, 1):
    print(f"{i}. {filename}")

เอกสารทั้งหมดในคลัง:
1. BIS_Ch01 Introduction to Business and Information Systems-69 (1).pdf
2. บทเรียนเรื่อง Python.txt
3. ประวัติของปัญญาประดิษฐ์ — จากยุคแรกสู่ LLM (1).pdf
4. ประวัติของปัญญาประดิษฐ์ — จากยุคแรกสู่ LLM.pdf
5. ใบงานแล็บสัปดาห์ที่ 4 เปิดสมุดพกและลงครัวปรุงข้อมูล.pdf
6. ใบงานแล็บสัปดาห์ที่ 5 สร้างสมองจิ๋วด้วยมือ.pdf


In [135]:
for filename in sorted(set(
    chunk["source"] for chunk in ALL_CHUNKS
)):
    if "ประวัติของปัญญาประดิษฐ์" in filename:
        chunks = [
            c for c in ALL_CHUNKS
            if c["source"] == filename
        ]
        print(filename)
        print("จำนวน Chunks:", len(chunks))
        print("ตัวอย่างเนื้อหา:")
        print(chunks[0]["text"][:300])
        print("-" * 50)

ประวัติของปัญญาประดิษฐ์ — จากยุคแรกสู่ LLM (1).pdf
จำนวน Chunks: 23
ตัวอย่างเนื้อหา:
ป ร ะ วัติศ า ส ต ร์ปัญ ญ า ป ร ะ ดิษ ฐ์
จากความฝันของนักคณิตศาสตร์  สู่
โมเดลภาษาขนาดยักษ์
เส้นทางกว่า  80 ป ของ  AI — ผ่านความหวัง  ความล้มเหลว  ฤดูหนาวที ยาวนาน  และ
การกลับมาที เปลี ยนโลก  ลําดับเหตุการณ์สําคัญที ทําให้เครื องจักรเริ ม  " คิด " และ
" พูด " ได้
นิวรอน โครงข่าย เรียนรู้เชิงลึก
 ภา
--------------------------------------------------
ประวัติของปัญญาประดิษฐ์ — จากยุคแรกสู่ LLM.pdf
จำนวน Chunks: 12
ตัวอย่างเนื้อหา:
ป ร ะ วัติศ า ส ต ร์ปัญ ญ า ป ร ะ ดิษ ฐ์
จากความฝันของนักคณิตศาสตร์  สู่
โมเดลภาษาขนาดยักษ์
เส้นทางกว่า  80 ป ของ  AI — ผ่านความหวัง  ความล้มเหลว  ฤดูหนาวที ยาวนาน  และ
การกลับมาที เปลี ยนโลก  ลําดับเหตุการณ์สําคัญที ทําให้เครื องจักรเริ ม  " คิด " และ
" พูด " ได้
นิวรอน โครงข่าย เรียนรู้เชิงลึก
 ภา
--------------------------------------------------


In [136]:
import re
import hashlib

ai_docs = [
    name for name in set(c["source"] for c in ALL_CHUNKS)
    if "ประวัติของปัญญาประดิษฐ์" in name
]

for filename in sorted(ai_docs):
    chunks = [
        c for c in ALL_CHUNKS
        if c["source"] == filename
    ]

    full_text = "\n".join(c["text"] for c in chunks)
    normalized = re.sub(r"\s+", "", full_text)

    fingerprint = hashlib.md5(
        normalized.encode("utf-8")
    ).hexdigest()

    print("ไฟล์:", filename)
    print("จำนวน Chunks:", len(chunks))
    print("จำนวนตัวอักษร:", len(normalized))
    print("Fingerprint:", fingerprint)
    print("-" * 40)

ไฟล์: ประวัติของปัญญาประดิษฐ์ — จากยุคแรกสู่ LLM (1).pdf
จำนวน Chunks: 23
จำนวนตัวอักษร: 9065
Fingerprint: 2af73bbf8a9d47e00aa6c466169e63cf
----------------------------------------
ไฟล์: ประวัติของปัญญาประดิษฐ์ — จากยุคแรกสู่ LLM.pdf
จำนวน Chunks: 12
จำนวนตัวอักษร: 8290
Fingerprint: c131fd1edc4dbbde5c2323d1f7db3d91
----------------------------------------


In [137]:
def clear_all_documents():
    global ALL_CHUNKS, ALL_VECTORS

    deleted_count = len(set(
        chunk["source"] for chunk in ALL_CHUNKS
    ))

    ALL_CHUNKS = []
    ALL_VECTORS = None

    return (
        f"🗑️ ล้างคลังเอกสารเรียบร้อย\n\n"
        f"- ลบเอกสารออกจากคลัง: {deleted_count} ไฟล์\n"
        f"- Chunks ที่เหลือ: {len(ALL_CHUNKS)}"
    )

print("ระบบล้างคลังพร้อมใช้งาน")

ระบบล้างคลังพร้อมใช้งาน


## Responsible AI — เจริญ (AI RAG Study Assistant)

### 1. ความเสี่ยงที่อาจเกิดขึ้น
- **Hallucination:** AI อาจสร้างคำตอบที่ไม่มีอยู่ในเอกสาร
- **Incorrect Retrieval:** ระบบอาจค้นพบเนื้อหาที่ไม่ตรงกับคำถาม
- **Incomplete Information:** เอกสารอาจมีข้อมูลไม่ครบถ้วน
- **Privacy:** เอกสารที่ผู้ใช้อัปโหลดอาจมีข้อมูลส่วนบุคคลหรือข้อมูลที่ไม่ควรเปิดเผย

### 2. วิธีลดความเสี่ยง
- กำหนดให้ AI ตอบโดยอ้างอิงข้อมูลจากเอกสารที่ค้นพบเท่านั้น
- หากไม่มีข้อมูลเพียงพอ ให้แจ้งว่าไม่พบข้อมูลในเอกสาร
- แสดงแหล่งที่มาของข้อมูลที่ใช้ตอบ
- ทดสอบระบบด้วยชุดคำถามที่มีทั้งคำถามที่ตอบได้และตอบไม่ได้
- หลีกเลี่ยงการอัปโหลดเอกสารที่มีข้อมูลส่วนตัวหรือข้อมูลลับ

### 3. ข้อจำกัดของระบบ
- ความถูกต้องขึ้นอยู่กับคุณภาพและเนื้อหาของเอกสาร
- PDF ที่เป็นภาพสแกนอาจไม่สามารถอ่านข้อความได้
- การค้นหาอาจไม่พบข้อมูลที่เกี่ยวข้อง หากแบ่ง Chunk หรือเลือกจำนวนผลลัพธ์ไม่เหมาะสม
- คำตอบจาก AI ยังควรได้รับการตรวจสอบก่อนนำไปใช้อ้างอิง

### 4. หลักการใช้งานอย่างรับผิดชอบ
เจริญเป็นเครื่องมือช่วยเรียนรู้ ไม่ใช่แหล่งข้อมูลที่รับประกันความถูกต้อง 100% ผู้ใช้ควรตรวจสอบคำตอบกับเอกสารต้นฉบับเสมอ